# Silver Pre-EDA — Deliverable 1.2.1

This notebook completes the Silver Pre-EDA portion of the Medallion Architecture.

**Purpose:**  
To perform the initial cleaning, missing-value review, structural preparation, and readiness checks needed before deeper Silver-layer exploratory analysis.  
This includes validating the dataset structure, identifying early data quality issues, confirming sensor feature availability, and exporting a clean Silver-layer dataset.

**Outputs:**  
- Cleaned `pump__silver__train.parquet` ready for downstream EDA and modeling  
- Silver feature registry (`pump__silver__feature_registry.json`)  
- Status flags (`is_anomaly`, `is_normal`) and schema-aligned metadata  
- Structural consistency checks needed for the statistical comparison described in Section C of the project proposal

This deliverable ensures that the Silver layer provides a stable, reproducible foundation for both Silver EDA (Deliverable 1.2.2) and Gold modeling.

## Overview

This notebook loads the Bronze output, performs Silver Pre-EDA checks, and prepares cleaned metadata and profiling inputs. It is part of the Silver portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook loads the Bronze output, performs Silver Pre-EDA checks, and prepares cleaned metadata and profiling inputs.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify how the Silver outputs prepare data quality, profiling, and feature context for Gold modeling.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/EDA_Notebook_Pump_Silver_01_PreEDA_code_reference.md`


In [89]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union, Pattern, Literal, Mapping, cast

from pathlib import Path
import yaml
import re
import os
import logging
import wandb

# Pands and Numpy
import pandas as pd 
import numpy as np 

# Plotting/Charts
import matplotlib.pyplot as plt 
import seaborn as sns 

# SKLearn Modules
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import pyarrow.parquet as pq
import pyarrow as pa

import hashlib


# Custom Utilities Module

# Path Module
from utils.core.paths import get_paths

# File IO Helper
from utils.core.file_io import (
    load_data, 
    save_data, 
    save_json, 
    load_json
)

# Logging Profiler
from utils.core.logging_profiler import profile_dataframe

# Logging Module
from utils.core.logging_setup import (
    configure_logging, 
    log_layer_paths,
)

# Weight and Biases
from utils.core.wandb_utils import finalize_wandb_stage

# Truth Store
from utils.core.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash, 
    load_truth_record_by_hash, 
    load_parent_truth_record_from_dataframe,
    extract_truth_hash,
    load_truth_record_by_hash,
    get_dataset_name_from_truth,
    get_truth_hash,
    get_pipeline_mode_from_truth,
)

# Database Helpers
from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

# Postgre Layer Helpers
from utils.database.layer_postgres import (
    read_layer_dataframe, 
    write_layer_dataframe, 
    prepare_layer_dataframe,
)

# SQL Helpers
from utils.database.sql_notebook_helpers import (
    delete_dataset_run_rows,
    execute_many,
    get_existing_dataframe,
    get_row_value,
    log_data_quality_event,
    log_pipeline_artifact,
    preview_sql_table,
    row_to_payload,
    sql_table_ref,
    to_builtin,
    to_json_string,
    to_scalar,
    upsert_pipeline_run,
)

# Layer Specific SQL Helper
from utils.database.medallion_sql_writers import (
    log_gold_05_anomaly_detection_summary_sql,
    log_silver_eda_sql,
    write_bronze_sensor_observations_sql,
    write_gold_baseline_scores_sql,
    write_gold_cascade_scores_sql,
    write_gold_model_comparison_results_sql,
    write_gold_preprocessed_features_sql,
    write_silver_sensor_observation_features_sql,
)

# Config Loader
from utils.core.config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

# Artifact Directory Creation
from utils.core.artifacts import (
    build_artifact_dirs,
    build_artifact_dirs_from_config,
    artifact_file_path,
)


# Ledger Loader
from utils.core.ledger import Ledger

# Notebook Context Loader
from utils.core.notebook_context import load_notebook_context


# Configuration Options

# JupyterLabs - Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# Literals Config
DropKeep = Literal["first", "last", False]



----

####  Helper Functions

In [90]:
def cfg_require_mapping(value: object, name: str) -> Mapping[str, Any]:
    if not isinstance(value, Mapping):
        raise TypeError(
            f"{name} must be a mapping, got {type(value).__name__}: {value!r}"
        )

    return cast(Mapping[str, Any], value)


def cfg_optional_mapping(value: object | None, name: str) -> Mapping[str, Any]:
    if value is None:
        return {}

    return cfg_require_mapping(value, name)

---

## Environment Setup, Paths, and Runtime Configuration

In this section I am loading the project paths and the resolved configuration values for the Silver Pre-EDA stage.

The goal here is to define the settings that control how this notebook runs before I start modifying the dataset itself. That includes:
- resolved project folders
- file names and output locations
- version values
- pipeline mode and run mode
- cleaning and truth-store settings used later in the notebook

I like setting this up early because it keeps the rest of the notebook more organized. Instead of hard-coding paths and runtime values throughout the workflow, I can define them once here and reuse them in a consistent way.

In [91]:
# Shared notebook context
CONTEXT_STAGE = "silver_preeda"
CONTEXT_DATASET = "pump"
CONTEXT_LAYER = "silver"
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"
CONTEXT_LOG_FILE = "silver.log"

CTX = load_notebook_context(
    stage=CONTEXT_STAGE,
    dataset=CONTEXT_DATASET,
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    logger_child_name="capstone.silver.preeda",
    log_filename=CONTEXT_LOG_FILE,
)

# Shared aliases used throughout the notebook
paths = CTX.paths
CONFIG = CTX.config
CONFIG_MAP = CTX.config
STAGE_CFG = CTX.stage_config
SILVER_CFG = CTX.stage_config
RESOLVED_PATHS = CTX.resolved_paths
FILENAMES = CTX.filenames
VERSIONS_CFG = CTX.versions
RUNTIME_CFG = CTX.runtime
DATASET_CFG = CTX.dataset_config
WANDB_CFG = CTX.wandb
EXECUTION_CFG = CTX.execution
PIPELINE = CTX.pipeline
logger = CTX.logger
ledger = CTX.ledger
LOG_PATH = CTX.log_path
CONTEXT_RECIPE_ID = CTX.recipe_id
DEFAULTS_FALLBACKS_CFG = CTX.default_fallbacks
DEFAULT_FALLBACKS_CFG = CTX.default_fallbacks

logger.info(
    "Notebook context loaded",
    extra={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
    },
)

ledger.add(
    kind="step",
    step="context_loaded",
    message="Loaded shared notebook context.",
    data={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)


2026-06-14 22:03:29,957 | INFO | capstone.silver.preeda | silver_preeda stage starting
2026-06-14 22:03:29,959 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:29.959934+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger from shared notebook context', 'why': None, 'consequence': None, 'data': {'stage': 'silver_preeda', 'recipe_id': 'silver__clean__dataset__agnostic__v001', 'dataset': 'pump', 'mode': 'train', 'profile': 'default', 'log_path': '/workspace/logs/silver.log'}}
2026-06-14 22:03:29,963 | INFO | capstone.silver.preeda | Notebook context loaded
2026-06-14 22:03:29,966 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:29.966050+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'context_loaded', 'message': 'Loaded shared notebook context.', 'why': None, 'consequence': None, 'data'

{'ts_utc': '2026-06-14T22:03:29.966050+00:00',
 'stage': 'silver_preeda',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'step',
 'step': 'context_loaded',
 'message': 'Loaded shared notebook context.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'silver_preeda',
  'recipe_id': 'silver__clean__dataset__agnostic__v001',
  'dataset': 'pump',
  'mode': 'train',
  'profile': 'default',
  'log_path': '/workspace/logs/silver.log'}}

In [92]:

TRUTH_CONFIG = cast(Dict[str, Any], build_truth_config_block(CONFIG))
TRUTH_CONFIG["pipeline"] = PIPELINE

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

# ---- Stage details ----
STAGE = "silver"

LAYER_NAME = str(SILVER_CFG["layer_name"])

CLEANING_RECIPE_ID = str(SILVER_CFG["cleaning_recipe_id"])

SILVER_VERSION = str(VERSIONS_CFG["silver"])
TRUTH_VERSION = str(VERSIONS_CFG["truth"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

PIPELINE_MODE = str(PIPELINE["execution_mode"])
RUN_MODE = str(RUNTIME_CFG.get("mode", CONFIG_RUN_MODE))

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

DATASET_NAME_CONFIG = str(DATASET_CFG.get("name", "pump"))
DATASET_NAME = DATASET_NAME_CONFIG

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

SILVER_PROCESS_RUN_ID = make_process_run_id(
    str(SILVER_CFG["process_run_id_prefix"])
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- W&B ----
WANDB_PROJECT = str(WANDB_CFG.get("project", "capstone"))
WANDB_ENTITY = str(WANDB_CFG.get("entity", ""))
WANDB_RUN_NAME = f"{SILVER_VERSION}"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Canonical outputs ----
CANONICAL_OUTPUT_COLUMNS = list(SILVER_CFG["canonical_output_columns"])
CANONICAL_NON_META_ORDER = list(SILVER_CFG["canonical_non_meta_order"])

META_REQUIRED_COLUMNS = list(SILVER_CFG["meta_required_columns"])

CANONICAL_EXCLUDE_COLUMNS = list(SILVER_CFG["canonical_exclude_columns"])
LABEL_EXCLUDE_COLUMNS = list(SILVER_CFG["label_exclude_columns"])

LABEL_COLUMNS_ORDER = list(SILVER_CFG["label_columns_order"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Defaults / fallbacks ----


# ---- Defaults / fallbacks ----

ASSET_ID_DEFAULT_FALLBACK = str(
    os.getenv(
        "SYNTHETIC_ASSET_ID",
        DATASET_CFG.get(
            "asset_id",
            DEFAULTS_FALLBACKS_CFG.get("asset_id", "pump_asset_001"),
        ),
    )
)

RUN_ID_DEFAULT_FALLBACK = str(
    os.getenv(
        "SYNTHETIC_RUN_ID",
        DATASET_CFG.get(
            "run_id",
            DEFAULTS_FALLBACKS_CFG.get("run_id", "synthetic_run_001"),
        ),
    )
)

DATASET_RUN_ID = str(DATASET_CFG.get("run_id", RUN_ID_DEFAULT_FALLBACK))

RAW_PREFIX = str(SILVER_CFG["raw_prefix"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Candidate lists ----
TIME_COLUMN_CANDIDATES = list(SILVER_CFG["time_column_candidates"])
STEP_COLUMN_CANDIDATES = list(SILVER_CFG["step_column_candidates"])
TIE_BREAKER_CANDIDATES = list(SILVER_CFG["tie_breaker_candidates"])

STATUS_COLUMN_CANDIDATES = list(SILVER_CFG["status_column_candidates"])
LABEL_COLUMN_CANDIDATES = list(SILVER_CFG["label_column_candidates"])

NORMAL_STATUS_VALUES = {
    str(value)
    for value in DATASET_CFG.get("normal_status_values", ["normal"])
}

RegexLike = Union[str, Pattern[str]]
UNNAMED_COLUMN_REGEX = re.compile(
    r"^unnamed:\s*\d+(\.\d+)?$",
    flags=re.IGNORECASE,
)

JUNK_COLUMN_CANDIDATES = [
    "Unnamed:",
    "Unnamed",
    "unnamed",
    "...",
    "level_0",
    "",
    " ",
    "\t",
    "\ufeff",
    "\ufeff<name>",
]

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- QA / EDA thresholds ----
MIN_TIME_PARSE_SUCCESS_PERCENT = float(SILVER_CFG["min_time_parse_success_percent"])
MIN_STEP_PARSE_SUCCESS_PERCENT = float(SILVER_CFG["min_step_parse_success_percent"])

MISSING_STATE_DIFF_GATE_PCT = float(SILVER_CFG["minimum_state_difference_gate_percentage"])
MIN_ROWS_PER_STATE_FOR_GATE = int(SILVER_CFG["minimum_rows_per_state_for_gate"])
MISSING_SYNTHETIC_SUFFIX = str(SILVER_CFG["missingness_synthetic_suffix"])

QUARANTINE_MISSING_PCT = float(SILVER_CFG["quarantine_missing_pct"])
CORRELATION_THRESHOLD = float(SILVER_CFG["correlation_threshold"])

TOP_N_SENSORS_FOR_PLOTS = int(SILVER_CFG["top_n_sensors_for_plots"])
PAIRPLOT_SENSOR_CAP = int(SILVER_CFG["pairplot_sensor_cap"])
PAIRPLOT_SAMPLE_N = int(SILVER_CFG["pairplot_sample_n"])
TOP_PLOT_COLS = int(SILVER_CFG["top_plot_cols"])
TOP_CORR_COLS = int(SILVER_CFG["top_corr_cols"])

ROLLING_MINUTES = int(SILVER_CFG["rolling_minutes"])
LOOKBACK_HOURS = int(SILVER_CFG["lookback_hours"])
BASELINE_DAYS = int(SILVER_CFG["baseline_days"])
BASELINE_GAP_HOURS = int(SILVER_CFG["baseline_gap_hours"])
SUSTAIN_MINUTES = int(SILVER_CFG["sustain_minutes"])
TOP_SENSOR_PRE_HOURS = int(SILVER_CFG["top_sensor_pre_hours"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- File names ----
BRONZE_TRAIN_DATA_FILE_NAME = str(FILENAMES["bronze_train_file_name"])
SILVER_TRAIN_DATA_FILE_NAME = str(FILENAMES["silver_train_file_name"])
DROPPED_SENSORS_FILE_NAME = str(FILENAMES["silver_preeda_dropped_sensors_file_name"])

FEATURE_REGISTRY_FILE_NAME = str(
    FILENAMES.get("feature_registry_file_name", "feature_registry.json")
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Base paths only ----
# Artifact directories are created later, after DATASET_NAME is resolved from the Bronze parent truth.
BRONZE_TRAIN_DATA_PATH = Path(str(RESOLVED_PATHS["data_bronze_train_dir"]))
SILVER_TRAIN_DATA_PATH = Path(str(RESOLVED_PATHS["data_silver_train_dir"]))

TRUTHS_PATH = Path(str(RESOLVED_PATHS["truths_dir"]))
TRUTH_INDEX_PATH = Path(str(RESOLVED_PATHS["truth_index_path"]))
LOGS_PATH = Path(str(RESOLVED_PATHS["logs_root"]))

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# W&B
set_wandb_dir_from_config(CONFIG)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# Path failsafes
SILVER_TRAIN_DATA_PATH.mkdir(parents=True, exist_ok=True)
TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:", paths.root)
print("Artifacts root:", paths.artifacts)
print("Bronze input dir:", BRONZE_TRAIN_DATA_PATH)
print("Silver output dir:", SILVER_TRAIN_DATA_PATH)


Project root: /workspace
Artifacts root: /workspace/artifacts
Bronze input dir: /workspace/data/bronze/train
Silver output dir: /workspace/data/silver/train


----

#### Context Santity Check

In [ ]:
required_context_vars = [
    "CTX",
    "paths",
    "CONFIG",
    "CONFIG_MAP",
    "STAGE_CFG",
    "RESOLVED_PATHS",
    "FILENAMES",
    "VERSIONS_CFG",
    "RUNTIME_CFG",
    "DATASET_CFG",
    "WANDB_CFG",
    "EXECUTION_CFG",
    "PIPELINE",
    "logger",
    "ledger",
    "LOG_PATH",
]

missing_context_vars = [name for name in required_context_vars if name not in globals()]
if missing_context_vars:
    raise NameError(f"Missing required shared context variables: {missing_context_vars}")

logger.info(
    "Context sanity check passed",
    extra={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
)

ledger.add(
    kind="check",
    step="context_sanity_check",
    message="Verified shared notebook context variables are available.",
    data={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
    logger=logger,
)

pd.DataFrame(
    [
        {"name": name, "status": "present"}
        for name in required_context_vars
    ]
)

[]

['DEFAULT_FALLBACKS']

2026-06-14 22:03:30,098 | INFO | capstone.silver.preeda | Context sanity check
2026-06-14 22:03:30,101 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:30.101636+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'check', 'step': 'context_sanity_check', 'message': 'Verified shared notebook context variables are available.', 'why': None, 'consequence': None, 'data': {'stage': 'silver_preeda', 'recipe_id': 'silver__clean__dataset__agnostic__v001', 'dataset': 'pump', 'mode': 'train', 'profile': 'default', 'log_path': '/workspace/logs/silver.log'}}


{'ts_utc': '2026-06-14T22:03:30.101636+00:00',
 'stage': 'silver_preeda',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'check',
 'step': 'context_sanity_check',
 'message': 'Verified shared notebook context variables are available.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'silver_preeda',
  'recipe_id': 'silver__clean__dataset__agnostic__v001',
  'dataset': 'pump',
  'mode': 'train',
  'profile': 'default',
  'log_path': '/workspace/logs/silver.log'}}

In [ ]:
silver_required_context_vars = [
    "SILVER_CFG",
]

missing_silver_context_vars = [
    name for name in silver_required_context_vars
    if name not in globals()
]

if missing_silver_context_vars:
    raise NameError(f"Missing Silver context variables: {missing_silver_context_vars}")

logger.info("Silver context sanity check passed")

---


### Defer Silver artifact folder creation

The final Silver artifact folders should use DATASET_NAME from the Bronze parent truth record. At this point, DATASET_NAME has not been resolved yet.

The actual build_artifact_dirs(...) call happens after the Bronze parent truth is loaded and DATASET_NAME is confirmed.

In [94]:
SILVER_PREEDA_ARTIFACT_DIRS = {}

SILVER_ARTIFACTS_PATH = None
SILVER_REGISTRY_DIR = None
SILVER_PROFILE_DIR = None
SILVER_QUALITY_DIR = None
SILVER_SUMMARY_DIR = None
SILVER_METADATA_DIR = None
SILVER_CONFIG_DIR = None
SILVER_LINEAGE_DIR = None

CONFIG_SNAPSHOT_PATH = None
DROPPED_SENSORS_DATA_PATH = None
FEATURE_REGISTRY_PATH = None

print("Silver artifact folder creation deferred until DATASET_NAME is resolved.")


Silver artifact folder creation deferred until DATASET_NAME is resolved.


---

### SQL Runtime Context
Purpose:
Establish the Postgres connection and resolve the dataset/run identifiers used by SQL logging and medallion table writes.

In [95]:
# =============================================================================
# SQL Runtime Context
# Purpose:
#   Establish the Postgres connection and resolve the dataset/run identifiers
#   used by SQL logging and medallion table writes.
# =============================================================================

engine = get_engine_from_env()

CAPSTONE_SCHEMA: str = str(os.getenv("CAPSTONE_SCHEMA", "capstone"))


def first_non_empty_string(*values: object) -> str | None:
    """
    Return the first non-empty string-like value from a list of candidates.

    This helper skips None, empty strings, whitespace-only strings, and
    dictionaries. It is used to resolve dataset/run identifiers without
    accidentally accepting an invalid config object or a placeholder None value.
    """
    for value in values:
        if value is None:
            continue

        if isinstance(value, dict):
            continue

        text_value = str(value).strip()

        if text_value:
            return text_value

    return None


dataset_config = (
    cast(Dict[str, Any], CONFIG.get("dataset", {}))
    if isinstance(CONFIG.get("dataset", {}), dict)
    else {}
)

dataset_config_name = dataset_config.get("name")
dataset_config_id = dataset_config.get("dataset_id", dataset_config.get("id"))
dataset_config_run_id = dataset_config.get("run_id")
dataset_config_asset_id = dataset_config.get("asset_id")

is_synthetic_run = str(RUN_MODE).lower() in {
    "synthetic",
    "synthetic_train",
    "synthetic_run",
    "simulation",
}

if is_synthetic_run:
    raw_dataset_id = first_non_empty_string(
        os.getenv("CAPSTONE_DATASET_ID"),
        os.getenv("SYNTHETIC_DATASET_ID"),
        globals().get("DATASET_ID"),
        dataset_config_id,
        globals().get("DATASET_NAME_CONFIG"),
        dataset_config_name,
        "pump_synthetic_v1",
    )

    raw_run_id = first_non_empty_string(
        os.getenv("CAPSTONE_RUN_ID"),
        os.getenv("SYNTHETIC_RUN_ID"),
        globals().get("DATASET_RUN_ID"),
        dataset_config_run_id,
        globals().get("RUN_ID"),
        globals().get("RUN_ID_DEFAULT_FALLBACK"),
        "synthetic_run_001",
    )

    raw_asset_id = first_non_empty_string(
        os.getenv("CAPSTONE_ASSET_ID"),
        os.getenv("SYNTHETIC_ASSET_ID"),
        globals().get("ASSET_ID"),
        dataset_config_asset_id,
        globals().get("ASSET_ID_DEFAULT_FALLBACK"),
        "pump_asset_001",
    )

else:
    raw_dataset_id = first_non_empty_string(
        os.getenv("CAPSTONE_DATASET_ID"),
        globals().get("DATASET_ID"),
        dataset_config_id,
        globals().get("DATASET_NAME_CONFIG"),
        dataset_config_name,
        "pump",
    )

    raw_run_id = first_non_empty_string(
        os.getenv("CAPSTONE_RUN_ID"),
        globals().get("DATASET_RUN_ID"),
        dataset_config_run_id,
        globals().get("RUN_ID"),
        globals().get("RUN_ID_DEFAULT_FALLBACK"),
        "run__001",
    )

    raw_asset_id = first_non_empty_string(
        os.getenv("CAPSTONE_ASSET_ID"),
        globals().get("ASSET_ID"),
        dataset_config_asset_id,
        globals().get("ASSET_ID_DEFAULT_FALLBACK"),
        "asset__001",
    )


if raw_dataset_id is None:
    raise ValueError(
        "DATASET_ID could not be resolved. "
        "Set CAPSTONE_DATASET_ID or configure CONFIG['dataset']['name'] / "
        "CONFIG['dataset']['dataset_id']."
    )

if raw_run_id is None:
    raise ValueError(
        "RUN_ID could not be resolved. "
        "Set CAPSTONE_RUN_ID, CONFIG['dataset']['run_id'], or default_fallbacks.run_id."
    )

if raw_asset_id is None:
    raise ValueError(
        "ASSET_ID could not be resolved. "
        "Set CAPSTONE_ASSET_ID, CONFIG['dataset']['asset_id'], or default_fallbacks.asset_id."
    )


DATASET_ID: str = raw_dataset_id
RUN_ID: str = raw_run_id
ASSET_ID: str = raw_asset_id


print(f"SQL schema: {CAPSTONE_SCHEMA}")
print(f"Dataset ID: {DATASET_ID}")
print(f"Run ID: {RUN_ID}")
print(f"Asset ID: {ASSET_ID}")
print(f"Synthetic run: {is_synthetic_run}")

SQL schema: capstone
Dataset ID: pump
Run ID: run__001
Asset ID: asset__001
Synthetic run: False



### SQL Smoke Check

Purpose:
Confirm the database connection and expected capstone schemas/tables exist.


In [96]:
sql_smoke_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN (:capstone_schema, 'bronze', 'silver', 'gold', 'metadata')
    ORDER BY table_schema, table_name
    """,
    params={"capstone_schema": CAPSTONE_SCHEMA},
)

display(sql_smoke_check_dataframe)

,table_schema,table_name
0,bronze,sensor_observations
1,capstone,bronze_observations_input_stage
2,capstone,data_quality_events
3,capstone,pipeline_artifacts
4,capstone,pipeline_runs
5,capstone,simulation_state_control
6,capstone,simulation_timing_config
7,capstone,synthetic_observations_premelt_stage
8,capstone,synthetic_observations_timestamped_stage
9,capstone,synthetic_pump_stream


----

## Logging Setup

Before I start working on the Silver dataframe, I want logging turned on so this notebook records what happened during the run.

This helps with debugging and traceability. If a later step fails, or if I need to verify what inputs and outputs were used, the log file gives me a cleaner record of the notebook activity than relying on notebook output alone.

In [97]:
log_layer_paths(paths, current_layer=CONTEXT_LAYER, logger=logger)

logger.info(
    "Project paths logged for %s layer",
    CONTEXT_LAYER,
    extra={"stage": CONTEXT_STAGE, "layer": CONTEXT_LAYER},
)

ledger.add(
    kind="step",
    step=f"{CONTEXT_LAYER}_paths_logged",
    message="Logged project layer paths.",
    data={
        "stage": CONTEXT_STAGE,
        "layer": CONTEXT_LAYER,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)


2026-06-14 22:03:30,284 | INFO | capstone.silver.preeda | Project Root Path Loaded: /workspace
2026-06-14 22:03:30,287 | INFO | capstone.silver.preeda | Project Logging Path Loaded: /workspace/logs
2026-06-14 22:03:30,290 | INFO | capstone.silver.preeda | Project Artifacts Path Loaded: /workspace/artifacts
2026-06-14 22:03:30,292 | INFO | capstone.silver.preeda | Project Notebooks Path Loaded: /workspace/notebooks
2026-06-14 22:03:30,294 | INFO | capstone.silver.preeda | Project Truths Path Loaded: /workspace/artifacts/truths
2026-06-14 22:03:30,296 | INFO | capstone.silver.preeda | Project Data Path Loaded: /workspace/data
2026-06-14 22:03:30,299 | INFO | capstone.silver.preeda | Previous Layer (Bronze) Path Loaded: /workspace/data/bronze
2026-06-14 22:03:30,301 | INFO | capstone.silver.preeda | Previous Layer (Bronze) Training Path Loaded: /workspace/data/bronze/train
2026-06-14 22:03:30,303 | INFO | capstone.silver.preeda | Previous Layer (Bronze) Testing Path Loaded: /workspace/dat

{'ts_utc': '2026-06-14T22:03:30.317739+00:00',
 'stage': 'silver_preeda',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'step',
 'step': 'silver_paths_logged',
 'message': 'Logged project layer paths.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'silver_preeda',
  'layer': 'silver',
  'log_path': '/workspace/logs/silver.log'}}

In [98]:
'''
# Original Logging Setup

# Create silver log path 
silver_log_path = paths.logs / "silver.log"

# Initial Logger
configure_logging(
    "capstone",
    silver_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.silver")

# Log load and initiation
logger.info("Silver stage starting")

# Log paths loads
log_layer_paths(paths, current_layer="silver", logger=logger)
'''

'\n# Original Logging Setup\n\n# Create silver log path \nsilver_log_path = paths.logs / "silver.log"\n\n# Initial Logger\nconfigure_logging(\n    "capstone",\n    silver_log_path,\n    level=logging.DEBUG,\n    overwrite_handlers=True,\n)\n\n# Initiate Logger and log file\nlogger = logging.getLogger("capstone.silver")\n\n# Log load and initiation\nlogger.info("Silver stage starting")\n\n# Log paths loads\nlog_layer_paths(paths, current_layer="silver", logger=logger)\n'

----

## Initialize Experiment Tracking

Here I start the Weights & Biases run for the Silver stage.

I am using this to track the run context for this preprocessing notebook, such as version details, output locations, and key cleaning thresholds. This does not change the data itself. It is mainly part of the tracking and reproducibility side of the project.

In [99]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="silver",
    config={
        "silver_version": SILVER_VERSION,
        "cleaning_recipe_id": CLEANING_RECIPE_ID,
        "quarantine_missing_pct": QUARANTINE_MISSING_PCT,
        "min_time_parse_success_percent": MIN_TIME_PARSE_SUCCESS_PERCENT,
        "rolling_window": ROLLING_MINUTES,
        "bronze_path": str(BRONZE_TRAIN_DATA_PATH / BRONZE_TRAIN_DATA_FILE_NAME),
        "silver_out_dir": str(SILVER_TRAIN_DATA_PATH),
    },
)
logger.info("W&B initialized: %s", wandb_run.name)


2026-06-14 22:03:31,737 | INFO | capstone.silver.preeda | W&B initialized: silver__001


----

## Initialize the Silver Ledger

This notebook also uses a ledger to keep a structured record of important decisions and processing steps.

I treat the ledger as a cleaner summary of what happened during the notebook run. Instead of only having scattered notebook output, I also keep a machine-readable record of the major actions taken in Silver Pre-EDA.

In [100]:
# Ledger was initialized by load_notebook_context().
# Keep this cell as a visible notebook checkpoint instead of re-creating Ledger.

ledger.add(
    kind="step",
    step="ledger_context_ready",
    message="Ledger is available from shared notebook context.",
    data={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)

logger.info(
    "Ledger ready from notebook context",
    extra={"stage": CONTEXT_STAGE, "recipe_id": CONTEXT_RECIPE_ID},
)


2026-06-14 22:03:32,099 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:32.099588+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'ledger_context_ready', 'message': 'Ledger is available from shared notebook context.', 'why': None, 'consequence': None, 'data': {'stage': 'silver_preeda', 'recipe_id': 'silver__clean__dataset__agnostic__v001', 'log_path': '/workspace/logs/silver.log'}}
2026-06-14 22:03:32,102 | INFO | capstone.silver.preeda | Ledger ready from notebook context


In [101]:
'''
# Original Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=CLEANING_RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)
'''


'\n# Original Ledger Setup\n\nledger = Ledger(stage=STAGE, recipe_id=CLEANING_RECIPE_ID)\n\nledger.add(\n    kind="step",\n    step="init",\n    message="Initialized ledger",\n    logger=logger\n)\n'

----

## Load the Bronze Dataset into the Silver Workflow

This is the point where I bring the Bronze output into the Silver Pre-EDA notebook.

The Silver layer starts from the Bronze artifact, not from the original raw source directly. That matters because I want the Silver outputs to trace back to a known Bronze file that already carries dataset metadata and lineage information.

So in this step I am:
- locating the Bronze parquet file
- loading it into a dataframe
- recording the source path and shape
- previewing the first few rows before any Silver-specific cleaning begins

In [102]:
# Load Data

preferred_bronze = BRONZE_TRAIN_DATA_PATH / BRONZE_TRAIN_DATA_FILE_NAME

if preferred_bronze.exists():
    bronze_data_path = preferred_bronze
else:
    parquet_files = sorted(BRONZE_TRAIN_DATA_PATH.glob("*.parquet"))
    if len(parquet_files) == 0:
        raise FileNotFoundError(f"No parquet files found in {BRONZE_TRAIN_DATA_PATH}")
    if len(parquet_files) > 1: 
        logger.warning("Multiple Parquet Files found; Using First %s", parquet_files[0])
    bronze_data_path = parquet_files[0]

if not bronze_data_path.exists():
    raise FileNotFoundError(f"Bronze parquet not found: {bronze_data_path}")
    
dataframe = load_data(bronze_data_path.parent, bronze_data_path.name)



#### #### #### #### #### #### #### #### 

logger.info("Loaded Bronze: %s | shape=%s", bronze_data_path, dataframe.shape)
wandb_run.log({"bronze_rows": int(dataframe.shape[0]), "bronze_cols": int(dataframe.shape[1])})

ledger.add(
    kind="step",
    step="load_bronze",
    message="Loaded Bronze Parquet",
    why="Silver must be derived from reprodicible Bronze Artifact",
    consequence="All silver outputs trace back to this file",
    data={"bronze_path": str(bronze_data_path), "shape": list(dataframe.shape), "cols": len(dataframe.columns)},
    logger=logger
)


#### #### #### #### #### #### #### #### 

display(dataframe.head(3))

2026-06-14 22:03:32,780 | INFO | capstone.file_io | Loading Parquet: /workspace/data/bronze/train/pump__bronze__train.parquet


2026-06-14 22:03:33,638 | INFO | capstone.silver.preeda | Loaded Bronze: /workspace/data/bronze/train/pump__bronze__train.parquet | shape=(220320, 66)
2026-06-14 22:03:33,641 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:33.641682+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'load_bronze', 'message': 'Loaded Bronze Parquet', 'why': 'Silver must be derived from reprodicible Bronze Artifact', 'consequence': 'All silver outputs trace back to this file', 'data': {'bronze_path': '/workspace/data/bronze/train/pump__bronze__train.parquet', 'shape': [220320, 66], 'cols': 66}}


,meta__dataset,meta__split,meta__run_id,meta__asset_id,meta__ingested_at_utc,meta__source_file,meta__source_row_id,meta__record_id,meta__truth_hash,meta__parent_truth_hash,meta__pipeline_mode,Unnamed: 0,timestamp,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,machine_status
0,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,0,14598431322315673869,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,0,2018-04-01 00:00:00,2.465394,47.09201,53.2118,46.31076,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,NaN,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
1,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,1,15954729095895098000,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,1,2018-04-01 00:01:00,2.465394,47.09201,53.2118,46.31076,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,NaN,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
2,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,2,10041703297090838359,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,2,2018-04-01 00:02:00,2.444734,47.35243,53.2118,46.39757,638.8889,73.54598,13.32465,16.03733,15.61777,15.01013,37.86777,48.17723,32.08894,1.708474,420.8480,NaN,462.7798,459.6364,2.500062,666.2234,399.9418,880.4237,501.3617,982.7342,631.1326,740.8031,849.8997,454.2390,778.5734,715.6266,661.5740,721.8750,694.7721,441.2635,169.9820,343.1955,200.9694,93.90508,41.40625,31.25000,69.53125,30.46875,31.770830,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,241.3194,203.7037,NORMAL


----

## Resolve the Parent Truth Record and Confirm Dataset Identity

Before making Silver-layer changes, I want to confirm what Bronze artifact this dataframe came from and which dataset identity it belongs to.

This step connects the loaded dataframe back to its Bronze truth record. That gives me the parent truth hash, the dataset name, and the pipeline mode that should carry forward into the Silver layer.

I do this early because Silver should inherit its identity from Bronze, not invent a new one.

In [103]:
SILVER_PARENT_TRUTH_HASH = extract_truth_hash(dataframe)

if SILVER_PARENT_TRUTH_HASH is None:
    raise ValueError("Silver input dataframe does not contain a readable meta__truth_hash value.")

BRONZE_DATASET_NAME = (
    dataframe["meta__dataset"]
    .dropna()
    .astype("string")
    .str.strip()
)

BRONZE_DATASET_NAME = BRONZE_DATASET_NAME[BRONZE_DATASET_NAME != ""]

if len(BRONZE_DATASET_NAME) == 0:
    raise ValueError("Silver input dataframe is missing usable meta__dataset values.")

BRONZE_DATASET_NAME = str(BRONZE_DATASET_NAME.iloc[0]).strip()

parent_truth = load_parent_truth_record_from_dataframe(
    dataframe=dataframe,
    truth_dir=TRUTHS_PATH,
    parent_layer_name="bronze",
    dataset_name=BRONZE_DATASET_NAME,
    column_name="meta__truth_hash",
)

DATASET_NAME = get_dataset_name_from_truth(parent_truth)

SILVER_PARENT_TRUTH_HASH = get_truth_hash(parent_truth)

PARENT_PIPELINE_MODE = get_pipeline_mode_from_truth(parent_truth)
if PARENT_PIPELINE_MODE is not None:
    PIPELINE_MODE = PARENT_PIPELINE_MODE

if "meta__pipeline_mode" not in dataframe.columns:
    dataframe["meta__pipeline_mode"] = PIPELINE_MODE
else:
    dataframe["meta__pipeline_mode"] = dataframe["meta__pipeline_mode"].fillna(PIPELINE_MODE)

silver_truth = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=DATASET_NAME,
    layer_name=LAYER_NAME,
    process_run_id=SILVER_PROCESS_RUN_ID,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=SILVER_PARENT_TRUTH_HASH,
)

silver_truth = update_truth_section(
    silver_truth,
    "config_snapshot",
    {
        "silver_version": SILVER_VERSION,
        "cleaning_recipe_id": CLEANING_RECIPE_ID,
        "dataset_name_config": DATASET_NAME_CONFIG,
        "dataset_name_parent_truth": DATASET_NAME,
        "run_id_default_fallback": RUN_ID_DEFAULT_FALLBACK,
        "quarantine_missing_pct": QUARANTINE_MISSING_PCT,
        "min_time_parse_success_percent": MIN_TIME_PARSE_SUCCESS_PERCENT,
        "min_step_parse_success_percent": MIN_STEP_PARSE_SUCCESS_PERCENT,
        "pipeline_mode": PIPELINE_MODE,
    },
)

silver_truth = update_truth_section(
    silver_truth,
    "runtime_facts",
    {
        "parent_layer_name": "bronze",
        "parent_truth_hash": SILVER_PARENT_TRUTH_HASH,
        "dataset_name_from_parent_truth": DATASET_NAME,
    },
)

logger.info("Resolved Bronze parent truth hash: %s", SILVER_PARENT_TRUTH_HASH)
logger.info("Resolved Silver dataset name from Bronze truth: %s", DATASET_NAME)

print("Silver parent truth hash:", SILVER_PARENT_TRUTH_HASH)
print("Silver dataset name from parent truth:", DATASET_NAME)

2026-06-14 22:03:34,481 | INFO | capstone.silver.preeda | Resolved Bronze parent truth hash: b90b025534b790481484fe4870e5bff95c50143212898e9ff02b88401303da69
2026-06-14 22:03:34,484 | INFO | capstone.silver.preeda | Resolved Silver dataset name from Bronze truth: pump


Silver parent truth hash: b90b025534b790481484fe4870e5bff95c50143212898e9ff02b88401303da69
Silver dataset name from parent truth: pump


## Create the Silver Pre-EDA Artifact Folders After Dataset Resolution

At this point the notebook has loaded the Bronze parent truth record and confirmed the real `DATASET_NAME` that Silver should inherit.

This is where the Silver artifact folders should be created. I am intentionally not creating these folders earlier from the config fallback name, because that can create the same kind of structure drift we just corrected in Bronze.

The artifact utility is still the standard method. The difference is that I am now calling it after the dataset identity is known, so the folders, config snapshot, dropped-sensor artifact, feature registry, ledger, and summaries all line up under the actual Silver dataset path.


In [104]:
if DATASET_NAME is None or str(DATASET_NAME).strip() == "":
    raise ValueError("DATASET_NAME must be resolved from the Bronze parent truth before creating Silver artifacts.")

# Keep SQL dataset id aligned with the resolved dataset unless an explicit
# external DATASET_ID was provided.
DATASET_ID = os.getenv("DATASET_ID", DATASET_NAME)

SILVER_PREEDA_ARTIFACT_DIRS = build_artifact_dirs(
    artifacts_root=paths.artifacts,
    stage="silver",
    dataset_name=DATASET_NAME,
    family="preeda",
    subdirs=[
        "registry",
        "profiles",
        "quality",
        "summaries",
        "metadata",
        "config",
        "lineage",
    ],
)

SILVER_ARTIFACTS_PATH = SILVER_PREEDA_ARTIFACT_DIRS["root"]
SILVER_REGISTRY_DIR = SILVER_PREEDA_ARTIFACT_DIRS["registry"]
SILVER_PROFILE_DIR = SILVER_PREEDA_ARTIFACT_DIRS["profiles"]
SILVER_QUALITY_DIR = SILVER_PREEDA_ARTIFACT_DIRS["quality"]
SILVER_SUMMARY_DIR = SILVER_PREEDA_ARTIFACT_DIRS["summaries"]
SILVER_METADATA_DIR = SILVER_PREEDA_ARTIFACT_DIRS["metadata"]
SILVER_CONFIG_DIR = SILVER_PREEDA_ARTIFACT_DIRS["config"]
SILVER_LINEAGE_DIR = SILVER_PREEDA_ARTIFACT_DIRS["lineage"]

CONFIG_SNAPSHOT_PATH = (
    SILVER_CONFIG_DIR
    / f"{DATASET_NAME}__silver_preeda__resolved_config.yaml"
)

DROPPED_SENSORS_DATA_PATH = (
    SILVER_PROFILE_DIR
    / DROPPED_SENSORS_FILE_NAME
)

FEATURE_REGISTRY_PATH = (
    SILVER_REGISTRY_DIR
    / FEATURE_REGISTRY_FILE_NAME
)

if bool(EXECUTION_CFG.get("save_config_snapshot", True)):
    export_config_snapshot(
        CONFIG,
        destination=CONFIG_SNAPSHOT_PATH,
    )

print("Silver artifact root:", SILVER_ARTIFACTS_PATH)
print("Config snapshot:", CONFIG_SNAPSHOT_PATH)
print("Dropped sensors path:", DROPPED_SENSORS_DATA_PATH)
print("Feature registry path:", FEATURE_REGISTRY_PATH)


Silver artifact root: /workspace/artifacts/silver/pump/preeda
Config snapshot: /workspace/artifacts/silver/pump/preeda/config/pump__silver_preeda__resolved_config.yaml
Dropped sensors path: /workspace/artifacts/silver/pump/preeda/profiles/pump__silver_preeda__dropped_sensors.parquet
Feature registry path: /workspace/artifacts/silver/pump/preeda/registry/pump__silver__feature_registry.json


----

## Initial Silver Input Review

Before I start cleaning or restructuring anything, I want a first look at the dataframe as it entered the Silver stage.

At this point I am checking:
- shape
- column types
- general dataframe structure
- basic numeric summaries

This is still an early-stage review. I am not making final modeling decisions here. I just want to confirm that the Bronze handoff looks structurally reasonable before I begin Silver-specific preparation steps.

In [105]:
print("=== Silver Pre-EDA: Data Overview ===")
print(f"Shape: {dataframe.shape[0]:,} rows × {dataframe.shape[1]} columns\n")

print("=== Column Types ===")
display(dataframe.dtypes.to_frame("dtype").head(20))

print("\n=== df.info() ===")
display(dataframe.info())

print("\n=== Basic Descriptive Statistics (numeric only) ===")
display(dataframe.describe().T)

=== Silver Pre-EDA: Data Overview ===
Shape: 220,320 rows × 66 columns

=== Column Types ===


,dtype
meta__dataset,category
meta__split,category
meta__run_id,object
meta__asset_id,object
meta__ingested_at_utc,"datetime64[us, UTC]"
meta__source_file,string[python]
meta__source_row_id,int64
meta__record_id,uint64
meta__truth_hash,object
meta__parent_truth_hash,object



=== df.info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220320 entries, 0 to 220319
Data columns (total 66 columns):
 #   Column                   Non-Null Count   Dtype              
---  ------                   --------------   -----              
 0   meta__dataset            220320 non-null  category           
 1   meta__split              220320 non-null  category           
 2   meta__run_id             220320 non-null  object             
 3   meta__asset_id           220320 non-null  object             
 4   meta__ingested_at_utc    220320 non-null  datetime64[us, UTC]
 5   meta__source_file        220320 non-null  string             
 6   meta__source_row_id      220320 non-null  int64              
 7   meta__record_id          220320 non-null  uint64             
 8   meta__truth_hash         220320 non-null  object             
 9   meta__parent_truth_hash  0 non-null       object             
 10  meta__pipeline_mode      220320 non-null  object             

None


=== Basic Descriptive Statistics (numeric only) ===


,count,mean,std,min,25%,50%,75%,max
meta__source_row_id,220320.0,1.101595e+05,6.360105e+04,0.000000e+00,5.507975e+04,1.101595e+05,1.652392e+05,2.203190e+05
meta__record_id,220320.0,9.224243e+18,5.322367e+18,1.873763e+14,4.610703e+18,9.220552e+18,1.381850e+19,1.844674e+19
Unnamed: 0,220320.0,1.101595e+05,6.360105e+04,0.000000e+00,5.507975e+04,1.101595e+05,1.652392e+05,2.203190e+05
sensor_00,210112.0,2.372221e+00,4.122274e-01,0.000000e+00,2.438831e+00,2.456539e+00,2.499826e+00,2.549016e+00
sensor_01,219951.0,4.759161e+01,3.296666e+00,0.000000e+00,4.631076e+01,4.813368e+01,4.947916e+01,5.672743e+01
sensor_02,220301.0,5.086739e+01,3.666820e+00,3.315972e+01,5.039062e+01,5.164930e+01,5.277777e+01,5.603299e+01
sensor_03,220301.0,4.375248e+01,2.418887e+00,3.164062e+01,4.283854e+01,4.422743e+01,4.531250e+01,4.822049e+01
sensor_04,220301.0,5.906739e+02,1.440239e+02,2.798032e+00,6.266204e+02,6.326389e+02,6.376157e+02,8.000000e+02
sensor_05,220301.0,7.339641e+01,1.729825e+01,0.000000e+00,6.997626e+01,7.557679e+01,8.091215e+01,9.999988e+01
sensor_06,215522.0,1.350154e+01,2.163736e+00,1.446759e-02,1.334635e+01,1.364294e+01,1.453993e+01,2.225116e+01


### Ask

What is this first review helping me confirm?

### Answer

This check helps me answer a simple but important question: **did the dataset arrive in Silver in a usable state?**

At this point I am looking for broad structural confirmation:
- the row and column counts make sense
- the dtypes are not obviously broken
- the numeric summary does not show something immediately suspicious
- the dataframe looks stable enough to move into cleaning and feature preparation

So this is not deep EDA yet. It is more of a readiness check before I start applying Silver-layer rules.

----

## Remove Junk Import Columns

This step handles cleanup for import-related columns that should not be part of the real dataset.

These are usually columns created by file export or import issues, such as unnamed index-style columns or other placeholder fields that do not carry useful analytical information.

I remove them here so they do not interfere with later feature selection, schema checks, or saved outputs.

In [106]:
def remove_junk_import_columns(
        dataframe: pd.DataFrame, 
        *,
        junk_column_candidates: List[str],
        regex_pattern: Optional[RegexLike] = None,
        default_pattern: RegexLike = UNNAMED_COLUMN_REGEX,
    ) -> Tuple[pd.DataFrame, List[str], str]:

    dataframe_copy = dataframe.copy()
    
    def _ensure_regex_pattern(
            regex: Optional[RegexLike],
            *,
            default: RegexLike,
            flags: int = re.IGNORECASE,
        ) -> Pattern[str]:

        if regex is None:
            regex = default
        if isinstance(regex, re.Pattern):
            return regex
        if isinstance(regex, str):
            return re.compile(regex, flags=flags)

        raise TypeError(f"Regex must be a str, re.Pattern, or None; Instead got {type(regex)}")

    
    pattern = _ensure_regex_pattern(regex_pattern, default=default_pattern, flags=re.IGNORECASE)

    normalize_candidates = [
        str(candidate).strip().lower()
        for candidate in junk_column_candidates
        if candidate is not None and str(candidate).strip() != ""
        ]

    junk_prefixes = tuple(normalize_candidates)

    junk_columns_found: list[str] = []

    for column in dataframe_copy.columns:
        column_string = str(column)
        column_normalized = column_string.strip().lower()

        is_junk_prefix = column_normalized.startswith(junk_prefixes) if junk_prefixes else False
        is_regex_match = bool(pattern.search(column_string))

        if is_junk_prefix or is_regex_match:
            junk_columns_found.append(column_string)

    if junk_columns_found:
        dataframe_copy = dataframe_copy.drop(columns=junk_columns_found, errors="ignore")

    return dataframe_copy, junk_columns_found, pattern.pattern


#### Remove import-generated junk columns

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [107]:
dataframe, junk_columns_found, pattern_used = remove_junk_import_columns(
    dataframe,
    junk_column_candidates=JUNK_COLUMN_CANDIDATES,
    regex_pattern=UNNAMED_COLUMN_REGEX,
    default_pattern=UNNAMED_COLUMN_REGEX,
)

#### #### #### #### #### #### #### #### 

if junk_columns_found:

    logger.info("Dropped junk columns: %s", junk_columns_found)

    ledger.add(
        kind="step",
        step="remove_junk_import_columns",
        message="Dropped junk columns via candidates and/or regex pattern",
        data={"dropped": junk_columns_found, "pattern": pattern_used},
        logger=logger,
    )
else:
    logger.info("No junk columns found.")

    ledger.add(
        kind="step",
        step="remove_junk_import_columns",
        message="No junk columns matched candidates/regex pattern",
        data={"dropped": [], "pattern": pattern_used},
        logger=logger,
    )

#### #### #### #### #### #### #### #### 

2026-06-14 22:03:37,443 | INFO | capstone.silver.preeda | Dropped junk columns: ['Unnamed: 0']
2026-06-14 22:03:37,447 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:37.447423+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'remove_junk_import_columns', 'message': 'Dropped junk columns via candidates and/or regex pattern', 'why': None, 'consequence': None, 'data': {'dropped': ['Unnamed: 0'], 'pattern': '^unnamed:\\s*\\d+(\\.\\d+)?$'}}


----

## Remove Duplicate Column Names

After the first cleanup pass, I also check whether the dataframe contains duplicate column names.

Duplicate names can cause problems later during feature selection, saving, and truth stamping. So here I keep the first occurrence of each duplicated column and remove the extras to make sure the dataframe schema is unambiguous moving forward.

In [108]:
def deduplicate_columns(
    dataframe: pd.DataFrame,
    *,
    keep: DropKeep = "first",
) -> Tuple[pd.DataFrame, List[str]]:
    
    dataframe = dataframe.copy()
    
    if dataframe.columns.is_unique:
        return dataframe, []
    
    duplicates_found = dataframe.columns[dataframe.columns.duplicated()].tolist()

    dataframe = dataframe.loc[:, ~dataframe.columns.duplicated(keep=keep)]
    
    return dataframe, duplicates_found


#### Remove duplicate columns before canonicalization

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [109]:
dataframe, duplicates_columns_found = deduplicate_columns(dataframe, keep="first")

if duplicates_columns_found:
    logger.warning("Duplicate column names removed (kept first): %s", duplicates_columns_found)
else:
    logger.info("No duplicate column names detected.")

ledger.add(
    kind="step",
    step="deduplicate_columns",
    message="Removed duplicate column names (kept first occurrence).",
    data={"duplicates": duplicates_columns_found, "count": len(duplicates_columns_found)},
    logger=logger,
)

2026-06-14 22:03:38,385 | INFO | capstone.silver.preeda | No duplicate column names detected.
2026-06-14 22:03:38,387 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:38.387924+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'deduplicate_columns', 'message': 'Removed duplicate column names (kept first occurrence).', 'why': None, 'consequence': None, 'data': {'duplicates': [], 'count': 0}}


{'ts_utc': '2026-06-14T22:03:38.387924+00:00',
 'stage': 'silver_preeda',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'step',
 'step': 'deduplicate_columns',
 'message': 'Removed duplicate column names (kept first occurrence).',
 'why': None,
 'consequence': None,
 'data': {'duplicates': [], 'count': 0}}

----

## Validate the Dataset Name for Silver

Silver does not decide the dataset identity on its own. Instead, it verifies that the dataset name stamped in Bronze is present and still consistent.

This step confirms that the Silver dataframe is still tied to the correct dataset and that the dataset naming remains standardized. I want this validation in place before I continue with truth tracking and downstream exports.

In [110]:
def validate_dataset_name_for_silver(
        dataframe: pd.DataFrame,
        *,
        truth_dataset_name: Optional[str] = None,
        config_value: Optional[str] = None,
        bronze_source_column: str = "meta__dataset",
        fail_on_multiple_in_bronze: bool = True,
    ) -> Tuple[str, str, str]:
    """
    Validate dataset name for Silver.

    Silver does not resolve or assign dataset identity.
    Silver verifies that Bronze-stamped dataset identity exists and is consistent.
    """

    def _clean_values(series: pd.Series) -> pd.Series:
        values = (
            series.dropna()
            .astype("string")
            .str.strip()
        )
        return values[values != ""]

    def _normalize_dataset_name(dataset_name: str) -> str:
        normalized_value = str(dataset_name).strip().lower()
        normalized_value = normalized_value.replace(" ", "_")
        normalized_value = normalized_value.replace("-", "_")

        cleaned_characters = []
        for character in normalized_value:
            if character.isalnum() or character == "_":
                cleaned_characters.append(character)

        normalized_value = "".join(cleaned_characters)

        while "__" in normalized_value:
            normalized_value = normalized_value.replace("__", "_")

        normalized_value = normalized_value.strip("_")

        if normalized_value == "":
            raise ValueError("Dataset name normalization produced an empty value.")

        return normalized_value

    if bronze_source_column not in dataframe.columns:
        raise ValueError(
            f"Silver dataset validation failed because required column '{bronze_source_column}' is missing."
        )

    values = _clean_values(dataframe[bronze_source_column])

    if len(values) == 0:
        raise ValueError(
            f"Silver dataset validation failed because '{bronze_source_column}' exists but contains no usable values."
        )

    unique_values = values.unique()

    if len(unique_values) == 1:
        dataset_name = _normalize_dataset_name(str(unique_values[0]))
        dataset_method = "unique"
    else:
        if fail_on_multiple_in_bronze:
            raise ValueError(
                f"Silver dataset validation failed because multiple values were found in "
                f"'{bronze_source_column}'. Values discovered: {unique_values[:10]}"
            )
        top_value = values.value_counts().index[0]
        dataset_name = _normalize_dataset_name(str(top_value))
        dataset_method = "mode"

    if truth_dataset_name is not None and str(truth_dataset_name).strip() != "":
        truth_dataset_name_normalized = _normalize_dataset_name(str(truth_dataset_name))
        if dataset_name != truth_dataset_name_normalized:
            raise ValueError(
                "Silver dataset validation failed because dataframe meta__dataset does not match "
                f"truth dataset name. dataframe={dataset_name}, truth={truth_dataset_name_normalized}"
            )

    if config_value is not None and str(config_value).strip() != "":
        config_dataset_name_normalized = _normalize_dataset_name(str(config_value))
        if dataset_name != config_dataset_name_normalized:
            raise ValueError(
                "Silver dataset validation failed because dataframe meta__dataset does not match "
                f"configured dataset name. dataframe={dataset_name}, config={config_dataset_name_normalized}"
            )

    return dataset_name, bronze_source_column, dataset_method

#### Validate dataset identity for the Silver layer

This cell runs guardrail checks before the notebook continues. These checks reduce the risk of carrying missing columns, invalid labels, or inconsistent row counts into later outputs.

In [111]:
VALIDATED_DATASET_NAME, DATASET_SOURCE_COLUMN, DATASET_METHOD = validate_dataset_name_for_silver(
    dataframe=dataframe,
    truth_dataset_name=DATASET_NAME,
    config_value=None,
    bronze_source_column="meta__dataset",
    fail_on_multiple_in_bronze=True,
)

DATASET_NAME = str(VALIDATED_DATASET_NAME).strip().lower()

silver_truth["dataset_name"] = DATASET_NAME
silver_truth = update_truth_section(
    silver_truth,
    "runtime_facts",
    {
        "dataset_validation": {
            "dataset_name": DATASET_NAME,
            "dataset_source_column": DATASET_SOURCE_COLUMN,
            "dataset_method": DATASET_METHOD,
        }
    },
)

ledger.add(
    kind="decision",
    step="validate_dataset_name",
    message="Validated Bronze-stamped dataset name for Silver against parent truth and recorded the validation in Truth Store.",
    data={
        "dataset_name": DATASET_NAME,
        "dataset_source_col": DATASET_SOURCE_COLUMN,
        "dataset_method": DATASET_METHOD,
    },
    logger=logger,
)

2026-06-14 22:03:39,381 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:39.381661+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'decision', 'step': 'validate_dataset_name', 'message': 'Validated Bronze-stamped dataset name for Silver against parent truth and recorded the validation in Truth Store.', 'why': None, 'consequence': None, 'data': {'dataset_name': 'pump', 'dataset_source_col': 'meta__dataset', 'dataset_method': 'unique'}}


{'ts_utc': '2026-06-14T22:03:39.381661+00:00',
 'stage': 'silver_preeda',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'decision',
 'step': 'validate_dataset_name',
 'message': 'Validated Bronze-stamped dataset name for Silver against parent truth and recorded the validation in Truth Store.',
 'why': None,
 'consequence': None,
 'data': {'dataset_name': 'pump',
  'dataset_source_col': 'meta__dataset',
  'dataset_method': 'unique'}}

----

## Add and Confirm Required Silver Metadata Columns

Here I make sure the dataframe contains the required row-level metadata columns that Silver expects.

The point of this step is not to create brand new meaning for the data. It is to make sure the dataframe has the supporting columns needed for tracing, grouping, ordering, and downstream pipeline use.

Any Silver runtime context that belongs at the run level is stored in the truth record, while the dataframe itself keeps the row-level metadata that needs to travel with the data.

In [112]:
dataframe = dataframe.copy()

# Silver runtime context goes to truth, not dataframe
SILVER_PROCESSED_AT_UTC = pd.Timestamp.now(tz="UTC")

# Ensure required row-level/source columns exist
for column_name in META_REQUIRED_COLUMNS:
    if column_name not in dataframe.columns:
        if column_name == "meta__dataset":
            dataframe[column_name] = pd.NA
        elif column_name == "meta__split":
            dataframe[column_name] = "unsplit"
        elif column_name == "meta__run_id":
            dataframe[column_name] = RUN_ID_DEFAULT_FALLBACK
        elif column_name == "meta__asset_id":
            dataframe[column_name] = ASSET_ID_DEFAULT_FALLBACK
        elif column_name == "meta__source_file":
            dataframe[column_name] = ""
        elif column_name == "meta__source_row_id":
            dataframe[column_name] = pd.RangeIndex(start=0, stop=len(dataframe), step=1)
        else:
            dataframe[column_name] = pd.NA

silver_truth = update_truth_section(
    silver_truth,
    "runtime_facts",
    {
        "processed_at_utc": SILVER_PROCESSED_AT_UTC,
        "silver_version": SILVER_VERSION,
        "cleaning_recipe_id": CLEANING_RECIPE_ID,
    },
)

ledger.add(
    kind="step",
    step="silver_meta_contract",
    message="Ensured required row-level/source meta exists. Silver runtime metadata is being stored in Truth Store, not row-stamped.",
    data={
        "required_meta_columns": list(META_REQUIRED_COLUMNS),
        "processed_at_utc": SILVER_PROCESSED_AT_UTC.isoformat(),
        "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
    },
    logger=logger,
)

2026-06-14 22:03:39,902 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:39.902811+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'silver_meta_contract', 'message': 'Ensured required row-level/source meta exists. Silver runtime metadata is being stored in Truth Store, not row-stamped.', 'why': None, 'consequence': None, 'data': {'required_meta_columns': ['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__source_file', 'meta__source_row_id', 'meta__ingested_at_utc', 'meta__record_id'], 'processed_at_utc': '2026-06-14T22:03:39.902351+00:00', 'shape': {'rows': 220320, 'cols': 65}}}


{'ts_utc': '2026-06-14T22:03:39.902811+00:00',
 'stage': 'silver_preeda',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'step',
 'step': 'silver_meta_contract',
 'message': 'Ensured required row-level/source meta exists. Silver runtime metadata is being stored in Truth Store, not row-stamped.',
 'why': None,
 'consequence': None,
 'data': {'required_meta_columns': ['meta__dataset',
   'meta__split',
   'meta__run_id',
   'meta__asset_id',
   'meta__source_file',
   'meta__source_row_id',
   'meta__ingested_at_utc',
   'meta__record_id'],
  'processed_at_utc': '2026-06-14T22:03:39.902351+00:00',
  'shape': {'rows': 220320, 'cols': 65}}}

----

## Resolve the Label or Status Source

Before I can build anomaly-related fields, I need to determine which column should act as the label or status source.

Different datasets may use different column names for this. So instead of assuming one fixed name, I search through candidate columns and choose the best available source using a simple policy:
- use a label column if one exists
- otherwise fall back to a status column
- otherwise record that no usable source was found

This makes the preprocessing step more controlled and easier to reuse.

In [113]:
def resolve_label_or_status_source(
    dataframe: pd.DataFrame, 
    *,
    label_candidates: list[str], 
    status_candidates: list[str],
    top_n: int = 10,
) -> tuple[Optional[str], str, Dict[str, Any]]:
    
    # Helper function to get value counts from dataframe column
    # holds the top n amount and convert the the value count information into 
    # keys to string format so ledger serialization works well
    def _top_values(column: str) -> Dict[str, int]:
        value_counts = dataframe[column].value_counts(dropna=False).head(top_n)
        return {str(key): int(value) for key, value in value_counts.items()}


    def _column_info(column: str) -> Dict[str, Any]:
        total_rows = int(len(dataframe))
        non_null = int(dataframe[column].notna().sum())
        unique = int(dataframe[column].nunique(dropna=False))
        return {
            "top_values": _top_values(column),
            "unique_count": unique,
            "non_null_count": non_null,
            "total_row_count": total_rows,
            "coverage_pct": float((non_null / total_rows) * 100.0) if total_rows > 0 else 0.0,
        }

    found_labels: Dict[str, Any] = {}
    found_status: Dict[str, Any] = {}

    
    # Label Candidates
    for column in label_candidates:
        if column in dataframe.columns:
            found_labels[column] = _column_info(column)
        
    # Status Candidates
    for column in status_candidates:
        if column in dataframe.columns:
            found_status[column] = _column_info(column)


    # Choose primary source (policy: label preferred)
    if len(found_labels) > 0:
        chosen_column = next(iter(found_labels.keys()))
        chosen_type = "label"
        chosen_info = found_labels[chosen_column]
        chosen_from = "label_candidates"
    elif len(found_status) > 0:
        chosen_column = next(iter(found_status.keys()))
        chosen_type = "status"
        chosen_info = found_status[chosen_column]
        chosen_from = "status_candidates"
    else:
        return None, "none", {
            "chosen_from": "none",
            "found_labels": {},
            "found_status": {},
            "top_values": {},
            "unique_count": 0,
            "non_null_count": 0,
            "total_row_count": int(len(dataframe)),
            "coverage_pct": 0.0,
        }

    # Pack full context (so we know if both existed)
    info = {
        "chosen_from": chosen_from,
        "found_labels": found_labels,
        "found_status": found_status, 
        **chosen_info,                  
    }

    return chosen_column, chosen_type, info

#### Review current column layout

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [114]:
display(dataframe.columns)

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id', 'meta__record_id', 'meta__truth_hash',
       'meta__parent_truth_hash', 'meta__pipeline_mode', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09',
       'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23',
       'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37',
       'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51',
       'machine_status'],
      dtype='object')

#### Resolve the label source used for anomaly flags

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [115]:
LABEL_SOURCE_COLUMN, LABEL_SOURCE_TYPE, LABEL_SOURCE_INFO = resolve_label_or_status_source(
    dataframe,
    label_candidates=LABEL_COLUMN_CANDIDATES,
    status_candidates=STATUS_COLUMN_CANDIDATES,
    top_n=10,
)

found_label_columns = list(LABEL_SOURCE_INFO.get("found_labels", {}).keys())
found_status_columns = list(LABEL_SOURCE_INFO.get("found_status", {}).keys())

HAS_LABEL_CANDIDATES = int(len(found_label_columns) > 0)
HAS_STATUS_CANDIDATES = int(len(found_status_columns) > 0)

chosen_summary = {
    "top_values": LABEL_SOURCE_INFO.get("top_values", {}),
    "unique_count": LABEL_SOURCE_INFO.get("unique_count", 0),
    "non_null_count": LABEL_SOURCE_INFO.get("non_null_count", 0),
    "total_row_count": LABEL_SOURCE_INFO.get("total_row_count", int(len(dataframe))),
    "coverage_pct": LABEL_SOURCE_INFO.get("coverage_pct", 0.0),
}

silver_truth = update_truth_section(
    silver_truth,
    "runtime_facts",
    {
        "label_resolution": {
            "label_source_column": LABEL_SOURCE_COLUMN,
            "label_source_type": LABEL_SOURCE_TYPE,
            "label_source_info": LABEL_SOURCE_INFO,
            "has_label_candidates": HAS_LABEL_CANDIDATES,
            "has_status_candidates": HAS_STATUS_CANDIDATES,
            "chosen_summary": chosen_summary,
        }
    },
)

ledger.add(
    kind="decision",
    step="resolve_label_or_status_source",
    message="Resolved label/status source and wrote the resolution to Truth Store.",
    data={
        "label_source_col": LABEL_SOURCE_COLUMN,
        "label_source_type": LABEL_SOURCE_TYPE,
        "found_label_columns": found_label_columns,
        "found_status_columns": found_status_columns,
        "has_label_candidates": HAS_LABEL_CANDIDATES,
        "has_status_candidates": HAS_STATUS_CANDIDATES,
        "chosen_summary": chosen_summary,
    },
    logger=logger,
)

2026-06-14 22:03:40,984 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:40.984686+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'decision', 'step': 'resolve_label_or_status_source', 'message': 'Resolved label/status source and wrote the resolution to Truth Store.', 'why': None, 'consequence': None, 'data': {'label_source_col': 'machine_status', 'label_source_type': 'status', 'found_label_columns': [], 'found_status_columns': ['machine_status'], 'has_label_candidates': 0, 'has_status_candidates': 1, 'chosen_summary': {'top_values': {'NORMAL': 205836, 'RECOVERING': 14477, 'BROKEN': 7}, 'unique_count': 3, 'non_null_count': 220320, 'total_row_count': 220320, 'coverage_pct': 100.0}}}


{'ts_utc': '2026-06-14T22:03:40.984686+00:00',
 'stage': 'silver_preeda',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'decision',
 'step': 'resolve_label_or_status_source',
 'message': 'Resolved label/status source and wrote the resolution to Truth Store.',
 'why': None,
 'consequence': None,
 'data': {'label_source_col': 'machine_status',
  'label_source_type': 'status',
  'found_label_columns': [],
  'found_status_columns': ['machine_status'],
  'has_label_candidates': 0,
  'has_status_candidates': 1,
  'chosen_summary': {'top_values': {'NORMAL': 205836,
    'RECOVERING': 14477,
    'BROKEN': 7},
   'unique_count': 3,
   'non_null_count': 220320,
   'total_row_count': 220320,
   'coverage_pct': 100.0}}}

### Ask

Why does the notebook stop to resolve a label or status source here?

### Answer

This matters because later Silver steps depend on having a consistent way to identify normal vs anomalous behavior.

If I do not resolve that source carefully, then anything built afterward—such as `anomaly_flag`, `is_normal`, `is_anomaly`, or episode IDs—could be based on the wrong column or on inconsistent assumptions.

So this step is really answering: **what column should Silver trust as the behavioral reference point for anomaly-related logic?**

In [116]:
display(dataframe.columns)

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id', 'meta__record_id', 'meta__truth_hash',
       'meta__parent_truth_hash', 'meta__pipeline_mode', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09',
       'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23',
       'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37',
       'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51',
       'machine_status'],
      dtype='object')

#### Preview current dataframe rows

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [117]:
display(dataframe.head())

,meta__dataset,meta__split,meta__run_id,meta__asset_id,meta__ingested_at_utc,meta__source_file,meta__source_row_id,meta__record_id,meta__truth_hash,meta__parent_truth_hash,meta__pipeline_mode,timestamp,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,machine_status
0,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,0,14598431322315673869,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,2018-04-01 00:00:00,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,NaN,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
1,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,1,15954729095895098000,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,2018-04-01 00:01:00,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,NaN,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
2,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,2,10041703297090838359,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,2018-04-01 00:02:00,2.444734,47.35243,53.2118,46.397570,638.8889,73.54598,13.32465,16.03733,15.61777,15.01013,37.86777,48.17723,32.08894,1.708474,420.8480,NaN,462.7798,459.6364,2.500062,666.2234,399.9418,880.4237,501.3617,982.7342,631.1326,740.8031,849.8997,454.2390,778.5734,715.6266,661.5740,721.8750,694.7721,441.2635,169.9820,343.1955,200.9694,93.90508,41.40625,31.25000,69.53125,30.46875,31.770830,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,241.3194,203.7037,NORMAL
3,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,3,2072036352569063261,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,2018-04-01 00:03:00,2.460474,47.09201,53.1684,46.397568,628.1250,76.98898,13.31742,16.24711,15.69734,15.08247,38.57977,48.65607,31.67221,1.579427,420.7494,NaN,462.8980,460.8858,2.509521,666.0114,399.1046,878.8917,499.0430,977.7520,625.4076,739.2722,847.7579,474.8731,779.5091,690.4011,686.1111,754.6875,683.3831,446.2493,166.4987,343.9586,193.1689,101.04060,41.92708,31.51042,72.13541,30.46875,31.510420,40.88541,39.062500,64.81481,51.21528,38.194440,155.9606,66.84028,240.4514,203.1250,NORMAL
4,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,4,4365040424004714369,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,2018-04-01 00:04:00,2.445718,47.13541,53.2118,46.397568,636.4583,76.58897,13.35359,16.21094,15.69734,15.08247,39.48939,49.06298,31.95202,1.683831,419.8926,NaN,461.4906,468.2206,2.604785,663.2111,400.5426,882.5874,498.5383,979.5755,627.1830,737.6033,846.9182,408.8159,785.2307,704.6937,631.4814,766.1458,702.4431,433.9081,164.7498,339.9630,193.8770,101.70380,42.70833,31.51042,76.82291,30.98958,31.510420,41.40625,38.773150,65.10416,51.79398,38.773150,158.2755,66.55093,242.1875

----

## Protect Canonical Output Column Names

This step protects the standardized column names that Silver wants to create, such as the canonical time and event-order fields.

If the incoming dataframe already has a column with one of those names, I do not want to overwrite the original source data. So instead, I rename the source version with a `raw__` prefix and keep the canonical name available for the Silver-generated field.

That way I preserve the original information while still keeping a clean canonical schema.


In [118]:
def protect_canonical_output_names(
        dataframe: pd.DataFrame,
        *,
        canonical_output_columns: List[str],
        raw_prefix: str = "raw__",
    ) -> Tuple[pd.DataFrame, Dict[str, str]]:

    """
    If the input dataset already contains a canonical output name (event_time/event_step/time_index),
    rename the existing column to raw__<name> (or a unique variant) so we don't overwrite it.

    Returns:
      - updated dataframe
      - rename_map {original_name: new_name}
    """

    dataframe = dataframe.copy()

    rename_map: Dict[str, str] = {}

    for canonical_name in canonical_output_columns:
        if canonical_name not in dataframe.columns:
            continue

        base_new_name = f"{raw_prefix}{canonical_name}"
        new_name = base_new_name

        # This should ensure uniqueness
        counter = 2
        while new_name in dataframe.columns:
            new_name = f"{base_new_name}__{counter}"
            counter += 1

        dataframe = dataframe.rename(columns={canonical_name: new_name})
        rename_map[canonical_name] = new_name

    return dataframe, rename_map

#### Protect canonical metadata column names

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [119]:
# Run protection
dataframe, rename_map = protect_canonical_output_names(
    dataframe,
    canonical_output_columns=CANONICAL_OUTPUT_COLUMNS,
    raw_prefix = "raw__"
)

# json my rename map
rename_map_json = {str(key): str(value) for key, value in rename_map.items()}


#### #### #### #### #### #### #### #### 


if rename_map_json:
    ledger.add(
        kind="step",
        step="canonical_name_collision_protection",
        message="Renamed input columns that collide with canonical outputs (Policy A).",
        why="Prevent overwriting raw source columns when creating canonical outputs.",
        consequence="Original values preserved under raw__*; canonical outputs can be created safely.",
        data={
            "policy": "A",
            "canonical_outputs": list(CANONICAL_OUTPUT_COLUMNS),
            "raw_prefix": "raw__",
            "collisions": int(len(rename_map_json)),
            "renames": rename_map_json,
            "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
        },
        logger=logger,
    )
else:
    ledger.add(
        kind="step",
        step="canonical_name_collision_protection",
        message="No canonical-name collisions found (Policy A).",
        why="Confirm input does not contain columns that conflict with canonical outputs.",
        consequence="Canonical outputs can be created without renaming any source columns.",
        data={
            "policy": "A",
            "canonical_outputs": list(CANONICAL_OUTPUT_COLUMNS),
            "collisions": 0,
            "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
        },
        logger=logger,
    )

display(rename_map_json)


2026-06-14 22:03:42,731 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:42.731209+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'canonical_name_collision_protection', 'message': 'No canonical-name collisions found (Policy A).', 'why': 'Confirm input does not contain columns that conflict with canonical outputs.', 'consequence': 'Canonical outputs can be created without renaming any source columns.', 'data': {'policy': 'A', 'canonical_outputs': ['event_time', 'event_step', 'time_index'], 'collisions': 0, 'shape': {'rows': 220320, 'cols': 65}}}


{}

----

## Build the Canonical Identity and Ordering Fields

This is one of the key structural preparation steps in the Silver Pre-EDA notebook.

Here I am trying to standardize how the dataset identifies and orders observations by creating or resolving fields such as:
- asset identity
- run identity
- event time
- event step
- time index
- event ID

The main goal is to create a stable ordering and identity structure that later analysis and modeling steps can rely on, even if the original dataset column names are inconsistent.

In [120]:
def _pick_first_existing_candidate_column(dataframe: pd.DataFrame, candidates: List[str]) -> Optional[str]: 
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate
    return None


def _ensure_grouping_columns_exists(
    dataframe: pd.DataFrame,
    *,
    asset_column: str = "meta__asset_id",
    run_column: str = "meta__run_id",
    default_asset_value: str = "asset__single",
    default_run_value: str = "run__single",
) -> pd.DataFrame:
    if asset_column not in dataframe.columns:
        dataframe[asset_column] = default_asset_value
    if run_column not in dataframe.columns:
        dataframe[run_column] = default_run_value
    return dataframe


def evaluate_time_candidates(
    dataframe: pd.DataFrame,
    *,
    candidates: List[str],
    minimum_parse_success_percent: float,
) -> Dict[str, Any]:

    chosen_time_column = None
    chosen_parse_time_series = None
    chosen_parse_success_percent = None
    
    for candidate in candidates:
        if candidate not in dataframe.columns:
            continue

        parsed_time_series = pd.to_datetime(dataframe[candidate], errors="coerce", utc=True)

        if len(dataframe) == 0:
            parse_success_percent = 0.0
        else:
            parse_success_percent = float(parsed_time_series.notna().mean() * 100.0)

        if parse_success_percent >= minimum_parse_success_percent:
            chosen_time_column = candidate
            chosen_parse_time_series = parsed_time_series
            chosen_parse_success_percent = parse_success_percent
            break

    return {
        "time_column_used": chosen_time_column,
        "time_parse_success_percent": chosen_parse_success_percent,
        "parsed_time_series": chosen_parse_time_series,
    }


def evaluate_step_candidates(
    dataframe: pd.DataFrame,
    *,
    candidates: List[str],
    minimum_parse_success_percent: float,
) -> Dict[str, Any]:

    chosen_step_column = None
    chosen_parse_success_percent = None
    
    for candidate in candidates:
        if candidate not in dataframe.columns:
            continue

        numeric_series = pd.to_numeric(dataframe[candidate], errors="coerce")

        if len(dataframe) == 0:
            parse_success_percent = 0.0
        else:
            parse_success_percent = float(numeric_series.notna().mean() * 100.0)

        if parse_success_percent >= minimum_parse_success_percent:
            chosen_step_column = candidate
            chosen_parse_success_percent = parse_success_percent
            break

    return {
        "step_column_used": chosen_step_column,
        "step_parse_success_percent": chosen_parse_success_percent,
    }


def build_canonical_identity_and_order_master(
    dataframe: pd.DataFrame,
    *,
    dataset_name: str,
    time_candidates: List[str],
    step_candidates: List[str],
    tie_breaker_candidates: List[str],
    minimum_time_parse_success_percent: float = 95.0,
    minimum_step_parse_success_percent: float = 95.0,
    default_asset_id: str = "asset__single",
    default_run_id: str = "run__single",
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    
    dataframe_copy = dataframe.copy()

    dataframe_copy = _ensure_grouping_columns_exists(
        dataframe_copy,
        default_asset_value=default_asset_id,
        default_run_value=default_run_id,
    )

    group_columns = ["meta__asset_id", "meta__run_id"]

    tie_breaker_column_used = _pick_first_existing_candidate_column(
        dataframe_copy,
        candidates=tie_breaker_candidates,
    )

    time_evaluation = evaluate_time_candidates(
        dataframe_copy,
        candidates=time_candidates,
        minimum_parse_success_percent=minimum_time_parse_success_percent,
    )

    step_evaluation = evaluate_step_candidates(
        dataframe_copy,
        candidates=step_candidates,
        minimum_parse_success_percent=minimum_step_parse_success_percent,
    )

    time_column_used = time_evaluation["time_column_used"]
    time_parse_success_percent = time_evaluation["time_parse_success_percent"]
    parsed_time_series = time_evaluation["parsed_time_series"]

    step_column_used = step_evaluation["step_column_used"]
    step_parse_success_percent = step_evaluation["step_parse_success_percent"]

    if time_column_used is not None:
        ordering_mode = "time"
        dataframe_copy["event_time"] = parsed_time_series
    else:
        dataframe_copy["event_time"] = pd.NaT
        ordering_mode = "step" if step_column_used is not None else "row"

    sort_columns: List[str] = []
    sort_columns.extend(group_columns)

    if ordering_mode == "time":
        sort_columns.append("event_time")
    elif ordering_mode == "step":
        sort_columns.append(step_column_used)

    if tie_breaker_column_used is not None:
        sort_columns.append(tie_breaker_column_used)

    should_sort = bool(ordering_mode != "row" or tie_breaker_column_used is not None)

    if should_sort and len(sort_columns) > 0:
        dataframe_copy = dataframe_copy.sort_values(sort_columns, kind="mergesort").reset_index(drop=True)

    dataframe_copy["event_step"] = dataframe_copy.groupby(group_columns, dropna=False).cumcount()
    dataframe_copy["time_index"] = dataframe_copy["event_step"].astype("int64")

    dataframe_copy["meta__event_id"] = (
        str(dataset_name)
        + ":" + dataframe_copy["meta__asset_id"].astype(str)
        + ":" + dataframe_copy["meta__run_id"].astype(str)
        + ":" + dataframe_copy["event_step"].astype(str)
    )

    if pd.api.types.is_datetime64_any_dtype(dataframe_copy["event_time"]):
        dataframe_copy["event_date"] = dataframe_copy["event_time"].dt.floor("D")
    else:
        dataframe_copy["event_date"] = pd.NaT

    group_count = int(dataframe_copy[group_columns].drop_duplicates().shape[0])

    info = {
        "ordering_mode": ordering_mode,
        "group_columns": group_columns,
        "group_count": group_count,
        "rows": int(len(dataframe_copy)),
        "time_column_used": time_column_used,
        "time_parse_success_percent": time_parse_success_percent,
        "step_column_used": step_column_used,
        "step_parse_success_percent": step_parse_success_percent,
        "tie_breaker_column_used": tie_breaker_column_used,
        "sorted": bool(should_sort),
        "sort_columns": sort_columns,
    }

    return dataframe_copy, info

#### Build canonical row identity and ordering fields

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [121]:
dataframe, canonical_info = build_canonical_identity_and_order_master(
    dataframe,
    dataset_name=DATASET_NAME,
    time_candidates=TIME_COLUMN_CANDIDATES,
    step_candidates=STEP_COLUMN_CANDIDATES,
    tie_breaker_candidates=TIE_BREAKER_CANDIDATES,
    minimum_time_parse_success_percent=MIN_TIME_PARSE_SUCCESS_PERCENT,
    minimum_step_parse_success_percent=MIN_STEP_PARSE_SUCCESS_PERCENT,
    default_asset_id=ASSET_ID_DEFAULT_FALLBACK,
    default_run_id=RUN_ID_DEFAULT_FALLBACK,
)

silver_truth = update_truth_section(
    silver_truth,
    "runtime_facts",
    {
        "canonical_info": canonical_info,
    },
)

ledger.add(
    kind="decision",
    step="build_canonical_identity_and_order_master",
    message="Built canonical identity + ordering master and wrote the resolution to Truth Store.",
    data={
        "dataset_name": DATASET_NAME,
        "time_candidates": list(TIME_COLUMN_CANDIDATES),
        "step_candidates": list(STEP_COLUMN_CANDIDATES),
        "tie_breaker_candidates": list(TIE_BREAKER_CANDIDATES),
        **canonical_info,
        "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
    },
    logger=logger,
)

preview_columns = [
    "meta__asset_id", "meta__run_id",
    "event_time", "event_step", "time_index",
    "meta__event_id", "event_date",
]

preview_columns = [column for column in preview_columns if column in dataframe.columns]

display(dataframe[preview_columns].head(10), canonical_info)

2026-06-14 22:03:44,483 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:44.483801+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'decision', 'step': 'build_canonical_identity_and_order_master', 'message': 'Built canonical identity + ordering master and wrote the resolution to Truth Store.', 'why': None, 'consequence': None, 'data': {'dataset_name': 'pump', 'time_candidates': ['timestamp', 'event_time', 'time', 'datetime', 'date'], 'step_candidates': ['cycle', 'step', 'event_step', 'sample', 'time_index'], 'tie_breaker_candidates': ['meta__source_row_id', 'meta__record_id'], 'ordering_mode': 'time', 'group_columns': ['meta__asset_id', 'meta__run_id'], 'group_count': 1, 'rows': 220320, 'time_column_used': 'timestamp', 'time_parse_success_percent': 100.0, 'step_column_used': None, 'step_parse_success_percent': None, 'tie_breaker_column_used': 'meta__source_row_id', 'sorted': True, 'sort_columns': ['meta__asset_id', 'me

,meta__asset_id,meta__run_id,event_time,event_step,time_index,meta__event_id,event_date
0,asset__001,run__001,2018-04-01 00:00:00+00:00,0,0,pump:asset__001:run__001:0,2018-04-01 00:00:00+00:00
1,asset__001,run__001,2018-04-01 00:01:00+00:00,1,1,pump:asset__001:run__001:1,2018-04-01 00:00:00+00:00
2,asset__001,run__001,2018-04-01 00:02:00+00:00,2,2,pump:asset__001:run__001:2,2018-04-01 00:00:00+00:00
3,asset__001,run__001,2018-04-01 00:03:00+00:00,3,3,pump:asset__001:run__001:3,2018-04-01 00:00:00+00:00
4,asset__001,run__001,2018-04-01 00:04:00+00:00,4,4,pump:asset__001:run__001:4,2018-04-01 00:00:00+00:00
5,asset__001,run__001,2018-04-01 00:05:00+00:00,5,5,pump:asset__001:run__001:5,2018-04-01 00:00:00+00:00
6,asset__001,run__001,2018-04-01 00:06:00+00:00,6,6,pump:asset__001:run__001:6,2018-04-01 00:00:00+00:00
7,asset__001,run__001,2018-04-01 00:07:00+00:00,7,7,pump:asset__001:run__001:7,2018-04-01 00:00:00+00:00
8,asset__001,run__001,2018-04-01 00:08:00+00:00,8,8,pump:asset__001:run__001:8,2018-04-01 00:00:00+00:00
9,asset__001,run__001,2018-04-01 00:09:00+00:00,9,9,pump:asset__001:run__001:9,2018-04-01 00:00:00+00:00


{'ordering_mode': 'time',
 'group_columns': ['meta__asset_id', 'meta__run_id'],
 'group_count': 1,
 'rows': 220320,
 'time_column_used': 'timestamp',
 'time_parse_success_percent': 100.0,
 'step_column_used': None,
 'step_parse_success_percent': None,
 'tie_breaker_column_used': 'meta__source_row_id',
 'sorted': True,
 'sort_columns': ['meta__asset_id',
  'meta__run_id',
  'event_time',
  'meta__source_row_id']}

#### Review current column layout

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [122]:
display(dataframe.columns)

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id', 'meta__record_id', 'meta__truth_hash',
       'meta__parent_truth_hash', 'meta__pipeline_mode', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09',
       'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23',
       'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37',
       'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51',
       'machine_status', 'event_time', 'event_step',

## Build the Binary Anomaly Flag

Once the label or status source is resolved, I convert that information into a consistent binary field called `anomaly_flag`.

This helps standardize downstream logic because later steps do not need to keep handling many possible source formats. Instead, they can use one clear field where:
- `0` means normal
- `1` means anomaly

Depending on the source, this may come from direct labels or from status values that need to be interpreted.

In [123]:
ANOMALY_FLAG_COLUMN = "anomaly_flag"

#### Define label-to-binary normalization

This cell defines helper logic used by the surrounding `Build the Binary Anomaly Flag` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [124]:

def normalize_label_to_binary(
    series: pd.Series,
) -> Tuple[pd.Series, Dict[str, Any]]:
    """
    Convert a label-like series to 0/1.
    Handles common patterns:
      - already numeric 0/1
      - booleans
      - strings like "normal"/"anomaly", "ok"/"fail", "true"/"false", "yes"/"no"
    Unknown/unhandled values become NaN then filled as 0 (conservative).
    """

    raw_series = series.copy()

    # Try numeric first
    numeric_series = pd.to_numeric(raw_series, errors="coerce")

    # If numeric has enough coverage, use it
    numeric_non_null_percent = float(numeric_series.notna().mean() * 100.0) if len(raw_series) else 0.0

    if numeric_non_null_percent >= 95.0:
        # Normalize: anything > 0 -> 1, else 0
        binary_series = (numeric_series.fillna(0) > 0).astype("int64")
        info = {
            "method": "numeric_threshold_gt_0",
            "numeric_non_null_percent": numeric_non_null_percent,
        }
        return binary_series, info

    # Fall back to string normalization
    string_series = raw_series.astype("string").str.strip().str.lower()

    mapping = {
        "1": 1, "true": 1, "yes": 1, "y": 1, "anomaly": 1, "fault": 1, "fail": 1, "failed": 1, "broken": 1,
        "0": 0, "false": 0, "no": 0, "n": 0, "normal": 0, "ok": 0, "good": 0, "healthy": 0, "running": 0,
    }

    mapped = string_series.map(mapping)
    mapped_non_null_percent = float(mapped.notna().mean() * 100.0) if len(mapped) else 0.0

    # Conservative fill: unknowns become 0 (normal)
    binary_series = mapped.fillna(0).astype("int64")

    info = {
        "method": "string_mapping_conservative_fill",
        "mapped_non_null_percent": mapped_non_null_percent,
        "numeric_non_null_percent": numeric_non_null_percent,
        "mapping_keys_count": int(len(mapping)),
    }
    return binary_series, info



#### Define status-to-anomaly conversion

This cell defines helper logic used by the surrounding `Build the Binary Anomaly Flag` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [125]:

def build_anomaly_flag_from_status(
    series: pd.Series,
    *,
    normal_value: Optional[str] = None,
) -> Tuple[pd.Series, Dict[str, Any]]:
    """
    Convert a status-like series into anomaly_flag:
      anomaly_flag = 1 if status != normal_value else 0

    If normal_value is None, choose the mode (most frequent non-null).
    """
    raw_series = series.astype("string").str.strip()

    # Remove empty strings
    cleaned = raw_series.where(raw_series != "", other=pd.NA)

    # Determine normal value
    chosen_normal_value = normal_value
    if chosen_normal_value is None:
        non_null_values = cleaned.dropna()
        if len(non_null_values) == 0:
            chosen_normal_value = None
        else:
            chosen_normal_value = str(non_null_values.value_counts().index[0])

    if chosen_normal_value is None:
        # No usable normal value; default all normal (0)
        anomaly_flag = pd.Series(np.zeros(len(series), dtype=np.int64), index=series.index)
        info = {
            "method": "status_no_non_null_values",
            "normal_value": None,
        }
        return anomaly_flag, info

    anomaly_flag = (cleaned.astype("string") != str(chosen_normal_value)).fillna(False).astype("int64")

    info = {
        "method": "status_compare_to_normal_value",
        "normal_value": str(chosen_normal_value),
        "unique_status_count": int(cleaned.nunique(dropna=True)),
        "non_null_count": int(cleaned.notna().sum()),
    }
    return anomaly_flag, info


#### Define label-to-binary normalization

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [126]:

anomaly_build_info: Dict[str, Any] = {
    "source_type": LABEL_SOURCE_TYPE, 
    "source_column": LABEL_SOURCE_COLUMN
    }

if LABEL_SOURCE_TYPE == "label" and LABEL_SOURCE_COLUMN:
    anomaly_series, method_info = normalize_label_to_binary(dataframe[LABEL_SOURCE_COLUMN])
    dataframe[ANOMALY_FLAG_COLUMN] = anomaly_series
    anomaly_build_info.update(method_info)

elif LABEL_SOURCE_TYPE == "status" and LABEL_SOURCE_COLUMN:
    anomaly_series, method_info = build_anomaly_flag_from_status(dataframe[LABEL_SOURCE_COLUMN], normal_value=None)
    dataframe[ANOMALY_FLAG_COLUMN] = anomaly_series
    anomaly_build_info.update(method_info)

    # Optional state flags (handy for EDA)
    normal_value_text = anomaly_build_info.get("normal_value")
    if normal_value_text is not None:
        cleaned_status = dataframe[LABEL_SOURCE_COLUMN].astype("string").str.strip()
        dataframe["status_normal_value"] = str(normal_value_text)
        dataframe["is_normal"] = (cleaned_status == str(normal_value_text)).fillna(False).astype("int64")
        dataframe["is_anomaly"] = (dataframe[ANOMALY_FLAG_COLUMN] == 1).astype("int64")

else:
    # No label/status available: default to all normal
    dataframe[ANOMALY_FLAG_COLUMN] = np.zeros(len(dataframe), dtype=np.int64)
    anomaly_build_info.update({"method": "no_label_or_status_available_default_all_normal"})


# Basic summary for ledger
anomaly_counts = dataframe[ANOMALY_FLAG_COLUMN].value_counts(dropna=False).to_dict()
anomaly_counts_json = {str(key): int(value) for key, value in anomaly_counts.items()}

anomaly_build_info["anomaly_flag_counts"] = anomaly_counts_json
anomaly_build_info["anomaly_rate_percent"] = float((dataframe[ANOMALY_FLAG_COLUMN].mean() * 100.0)) if len(dataframe) else 0.0

#### #### #### #### #### #### #### #### 
ledger.add(
    kind="step",
    step="build_anomaly_flag",
    message="Built anomaly_flag from resolved label/status source.",
    why="Silver requires a consistent binary anomaly flag for segmentation, evaluation, and synthetic anomaly generation.",
    consequence="Downstream steps can rely on anomaly_flag (0/1) being present across datasets.",
    data={
        "label_source_column": LABEL_SOURCE_COLUMN,
        "label_source_type": LABEL_SOURCE_TYPE,
        **anomaly_build_info,
        "shape": {"rows": int(len(dataframe)), "columns": int(len(dataframe.columns))},
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

# # Quick preview
preview_columns = [
    column for column in [
        LABEL_SOURCE_COLUMN, 
        "status_normal_value", 
        "is_normal", 
        "is_anomaly", 
        "anomaly_flag"
    ] 
    if column and column in dataframe.columns
]

display(dataframe[preview_columns].head(12), anomaly_build_info)

2026-06-14 22:03:46,385 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:03:46.385335+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'build_anomaly_flag', 'message': 'Built anomaly_flag from resolved label/status source.', 'why': 'Silver requires a consistent binary anomaly flag for segmentation, evaluation, and synthetic anomaly generation.', 'consequence': 'Downstream steps can rely on anomaly_flag (0/1) being present across datasets.', 'data': {'label_source_column': 'machine_status', 'label_source_type': 'status', 'source_type': 'status', 'source_column': 'machine_status', 'method': 'status_compare_to_normal_value', 'normal_value': 'NORMAL', 'unique_status_count': 3, 'non_null_count': 220320, 'anomaly_flag_counts': {'0': 205836, '1': 14484}, 'anomaly_rate_percent': 6.5740740740740735, 'shape': {'rows': 220320, 'columns': 74}}}


,machine_status,status_normal_value,is_normal,is_anomaly,anomaly_flag
0,NORMAL,NORMAL,1,0,0
1,NORMAL,NORMAL,1,0,0
2,NORMAL,NORMAL,1,0,0
3,NORMAL,NORMAL,1,0,0
4,NORMAL,NORMAL,1,0,0
5,NORMAL,NORMAL,1,0,0
6,NORMAL,NORMAL,1,0,0
7,NORMAL,NORMAL,1,0,0
8,NORMAL,NORMAL,1,0,0
9,NORMAL,NORMAL,1,0,0


{'source_type': 'status',
 'source_column': 'machine_status',
 'method': 'status_compare_to_normal_value',
 'normal_value': 'NORMAL',
 'unique_status_count': 3,
 'non_null_count': 220320,
 'anomaly_flag_counts': {'0': 205836, '1': 14484},
 'anomaly_rate_percent': 6.5740740740740735}

#### Review current column layout

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [127]:
display(dataframe.columns)

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id', 'meta__record_id', 'meta__truth_hash',
       'meta__parent_truth_hash', 'meta__pipeline_mode', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09',
       'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23',
       'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37',
       'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51',
       'machine_status', 'event_time', 'event_step',

----

## Build Episode IDs from the Anomaly Signal

After creating the anomaly flag, I build an episode identifier that groups rows into broader anomaly windows.

The idea here is to create a cleaner unit for downstream analysis by identifying stretches of time that belong to the same anomaly period. That makes later checks easier when I want to summarize or compare behavior at more than just the single-row level.

In [128]:
def build_episode_ids_from_anomaly_flag(
    dataframe: pd.DataFrame,
    *,
    anomaly_flag_column: str = "anomaly_flag",
    group_columns: list[str] | None = None,
) -> pd.Series:
    """
    Build an episode id for each row in a time series dataset.

    An 'episode' is:
      normal -> anomaly_flag==1 (failure+recovery window) -> back to normal

    Each time we transition from anomaly (1) back to normal (0),
    the episode id increments by 1.

    Episodes are computed separately per group (meta__asset_id / meta__run_id if available).
    """

    if anomaly_flag_column not in dataframe.columns:
        raise ValueError(f"anomaly_flag_column '{anomaly_flag_column}' not found in dataframe.")

    working = dataframe.copy()

    # 1) Decide grouping columns (per asset/run)
    if group_columns is None:
        group_columns = []
        if "meta__asset_id" in working.columns:
            group_columns.append("meta__asset_id")
        if "meta__run_id" in working.columns:
            group_columns.append("meta__run_id")

    if not group_columns:
        # Single global group
        working["_tmp_group"] = 0
        group_columns = ["_tmp_group"]
        drop_tmp_group = True
    else:
        drop_tmp_group = False

    # 2) Decide time ordering column
    if "time_index" in working.columns:
        ordering_column = "time_index"
    elif "event_step" in working.columns:
        ordering_column = "event_step"
    else:
        ordering_column = None

    anomaly_series = working[anomaly_flag_column].fillna(0).astype(int)

    # 3) Container for episode ids
    episode_ids = pd.Series(index=working.index, dtype="int64")

    grouped = working.groupby(group_columns, dropna=False)

    for _, group_df in grouped:
        # Sort by time inside each asset/run
        if ordering_column is not None and ordering_column in group_df.columns:
            group_df = group_df.sort_values(by=ordering_column)

        idx = group_df.index

        current_episode = 0
        in_anomaly_window = False  # are we currently inside a failure+recovery window?

        for row_idx in idx:
            flag = anomaly_series.loc[row_idx]

            if flag == 1:
                # We are in an anomaly window
                in_anomaly_window = True
            else:  # flag == 0
                if in_anomaly_window:
                    # We just transitioned from anomaly back to normal -> new episode
                    current_episode += 1
                    in_anomaly_window = False

            episode_ids.loc[row_idx] = current_episode

    if drop_tmp_group:
        working.drop(columns=["_tmp_group"], inplace=True, errors="ignore")

    return episode_ids.astype("int64")

#### Build anomaly episode identifiers

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [129]:
logger.info("Building episode ids from anomaly_flag for Silver dataframe.")

dataframe["meta__episode_id"] = build_episode_ids_from_anomaly_flag(
    dataframe,
    anomaly_flag_column="anomaly_flag",
)


episode_counts = dataframe["meta__episode_id"].value_counts().sort_index()

logger.info("Episode id summary: %s", episode_counts.to_dict())


#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="silver_build_episode_ids",
    message="Built meta__episode_id for Silver dataframe using anomaly windows.",
    data={
        "episode_column": "meta__episode_id",
        "unique_episode_count": int(dataframe["meta__episode_id"].nunique()),
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 


2026-06-14 22:03:47,697 | INFO | capstone.silver.preeda | Building episode ids from anomaly_flag for Silver dataframe.
2026-06-14 22:04:07,014 | INFO | capstone.silver.preeda | Episode id summary: {0: 18100, 1: 9521, 2: 43010, 3: 7765, 4: 58035, 5: 4742, 6: 25343, 7: 53804}
2026-06-14 22:04:07,019 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:04:07.019551+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'silver_build_episode_ids', 'message': 'Built meta__episode_id for Silver dataframe using anomaly windows.', 'why': None, 'consequence': None, 'data': {'episode_column': 'meta__episode_id', 'unique_episode_count': 8}}


{'ts_utc': '2026-06-14T22:04:07.019551+00:00',
 'stage': 'silver_preeda',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'step',
 'step': 'silver_build_episode_ids',
 'message': 'Built meta__episode_id for Silver dataframe using anomaly windows.',
 'why': None,
 'consequence': None,
 'data': {'episode_column': 'meta__episode_id', 'unique_episode_count': 8}}

----

## Identify the Candidate Feature Set

Now I shift from schema preparation into deciding which columns should count as actual model-ready features.

This step excludes columns that should not be treated as features, such as:
- metadata columns
- raw-protection columns
- canonical helper columns
- label or status source columns
- obvious identifier-style columns

From there, I classify the remaining candidates by type and build the initial feature set. For this notebook, the default approach is conservative and mostly keeps numeric and boolean fields.

In [130]:
DEFAULT_EXCLUDE_PREFIXES = ["meta__", "raw__"]


#### Classify column types for feature selection

This cell defines helper logic used by the surrounding `Identify the Candidate Feature Set` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [131]:

def classify_column_type(series: pd.Series) -> str:
    if pd.api.types.is_bool_dtype(series):
        return "boolean"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"
    #if pd.api.types.is_categorical_dtype(series):
    if isinstance(series.dtype, pd.CategoricalDtype):
        return "categorical"
    if pd.api.types.is_string_dtype(series) or pd.api.types.is_object_dtype(series):
        return "text"
    return "other"


#### Define prefix-based feature exclusions

This cell defines helper logic used by the surrounding `Identify the Candidate Feature Set` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [132]:

def should_exclude_by_prefix(column_name: str, exclude_prefixes: List[str]) -> bool:
    for prefix in exclude_prefixes:
        if column_name.startswith(prefix):
            return True
    return False


#### Review current column layout

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [133]:
display(dataframe.columns)

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id', 'meta__record_id', 'meta__truth_hash',
       'meta__parent_truth_hash', 'meta__pipeline_mode', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09',
       'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23',
       'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37',
       'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51',
       'machine_status', 'event_time', 'event_step',

#### Define identifier-column exclusion heuristic

This cell defines helper logic used by the surrounding `Identify the Candidate Feature Set` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [134]:

def looks_like_identifier_column(
    dataframe: pd.DataFrame,
    *,
    column_name: str,
    unique_ratio_threshold: float = 0.50,
) -> bool:
    """
    Heuristic: exclude obvious identifier-like columns from features.
    Examples: *id*, *uuid*, *serial*, etc. or extremely high-cardinality columns.
    """
    lower_name = column_name.lower()

    identifier_keywords = ["id", "uuid", "guid", "serial", "record"]
    keyword_hit = False
    for keyword in identifier_keywords:
        if keyword in lower_name:
            keyword_hit = True
            break

    series = dataframe[column_name]
    total_rows = int(len(series))

    if total_rows == 0:
        return False

    unique_count = int(series.nunique(dropna=True))
    unique_ratio = float(unique_count / total_rows)

    # If it looks like an ID name AND has lots of unique values, treat it like an identifier
    if keyword_hit and unique_ratio >= unique_ratio_threshold:
        return True

    # If it is extremely high-cardinality text, treat as identifier/text signal rather than numeric feature
    if (pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)) and unique_ratio >= 0.80:
        return True

    return False


#### Classify column types for feature selection

This cell defines helper logic used by the surrounding `Identify the Candidate Feature Set` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [135]:

def identify_feature_set(
    dataframe: pd.DataFrame,
    *,
    exclude_prefixes: List[str],
    exclude_columns: List[str],
    label_source_column: Optional[str],
    include_categorical_features: bool = False,
    include_text_features: bool = False,
    include_datetime_features: bool = False,
) -> Tuple[List[str], Dict[str, List[str]], Dict[str, Any]]:
    """
    Returns:
      - selected_feature_columns: List[str]
      - feature_groups: Dict[group_name, List[str]]
      - info: Dict[str, Any] with counts and decisions
    """
    candidate_columns: List[str] = []
    for column_name in dataframe.columns:
        if should_exclude_by_prefix(column_name, exclude_prefixes):
            continue
        if column_name in exclude_columns:
            continue
        if label_source_column is not None and column_name == label_source_column:
            continue
        if looks_like_identifier_column(dataframe, column_name=column_name):
            continue
        candidate_columns.append(column_name)

    feature_groups: Dict[str, List[str]] = {
        "numeric": [],
        "boolean": [],
        "categorical": [],
        "text": [],
        "datetime": [],
        "other": [],
    }

    for column_name in candidate_columns:
        column_type = classify_column_type(dataframe[column_name])
        feature_groups[column_type].append(column_name)

    selected_feature_columns: List[str] = []
    # Default: numeric + boolean (safest across datasets)
    selected_feature_columns.extend(feature_groups["numeric"])
    selected_feature_columns.extend(feature_groups["boolean"])

    if include_categorical_features:
        selected_feature_columns.extend(feature_groups["categorical"])

    if include_text_features:
        selected_feature_columns.extend(feature_groups["text"])

    if include_datetime_features:
        selected_feature_columns.extend(feature_groups["datetime"])

    # Keep stable ordering
    selected_feature_columns = sorted([str(name) for name in selected_feature_columns])

    info: Dict[str, Any] = {
        "candidate_column_count": int(len(candidate_columns)),
        "selected_feature_count": int(len(selected_feature_columns)),
        "group_counts": {
            "numeric": int(len(feature_groups["numeric"])),
            "boolean": int(len(feature_groups["boolean"])),
            "categorical": int(len(feature_groups["categorical"])),
            "text": int(len(feature_groups["text"])),
            "datetime": int(len(feature_groups["datetime"])),
            "other": int(len(feature_groups["other"])),
        },
        "selection_policy": {
            "include_categorical_features": bool(include_categorical_features),
            "include_text_features": bool(include_text_features),
            "include_datetime_features": bool(include_datetime_features),
        },
    }

    return selected_feature_columns, feature_groups, info


#### Select model-ready candidate features

This cell performs the next transformation in `Identify the Candidate Feature Set`. Review the output or logged message before relying on this result downstream.

In [136]:

exclude_columns_combined = []
exclude_columns_combined.extend(CANONICAL_EXCLUDE_COLUMNS)
exclude_columns_combined.extend(LABEL_EXCLUDE_COLUMNS)

FEATURE_COLUMNS, FEATURE_GROUPS, FEATURE_INFO = identify_feature_set(
    dataframe,
    exclude_prefixes=DEFAULT_EXCLUDE_PREFIXES,
    exclude_columns=exclude_columns_combined,
    label_source_column=LABEL_SOURCE_COLUMN,
    include_categorical_features=False,
    include_text_features=False,
    include_datetime_features=False,
)



----

## Identify Columns That May Need One-Hot Encoding Later

Even though one-hot encoding does not happen in this notebook, I still want to identify which columns may need it later in Gold.

This helps carry forward a cleaner preprocessing plan. Silver Pre-EDA is not only about cleaning the current dataframe. It is also about leaving behind enough structured information so the later stages know what to do next.

In [137]:
def identify_one_hot_encoding_columns(
    silver_dataframe: pd.DataFrame,
    *,
    excluded_columns: list[str] | None = None,
) -> tuple[list[str], dict]:
    """
    Identify categorical columns that should be one-hot encoded later in Gold.

    Returns
    -------
    one_hot_encoding_columns : list[str]
        Ordered list of categorical columns to encode.
    one_hot_encoding_truths : dict
        Truth-file fields describing whether encoding is needed and which columns
        were selected.
    """
    working_excluded_columns = set(excluded_columns or [])

    categorical_columns = [
        column_name
        for column_name in silver_dataframe.columns
        if column_name not in working_excluded_columns
        and (
            pd.api.types.is_object_dtype(silver_dataframe[column_name])
            #or pd.api.types.is_categorical_dtype(silver_dataframe[column_name])
            or isinstance(silver_dataframe[column_name].dtype, pd.CategoricalDtype)
            or pd.api.types.is_bool_dtype(silver_dataframe[column_name])
        )
    ]

    categorical_columns = sorted(categorical_columns)

    one_hot_encoding_truths = {
        "needs_one_hot_encoding": bool(categorical_columns),
        "one_hot_encoding_columns": categorical_columns,
    }

    return categorical_columns, one_hot_encoding_truths

#### Apply feature exclusion rules

This cell performs the next transformation in `Identify Columns That May Need One-Hot Encoding Later`. Review the output or logged message before relying on this result downstream.

In [138]:
silver_one_hot_encoding_columns, silver_one_hot_encoding_truths = identify_one_hot_encoding_columns(
    silver_dataframe=dataframe,
    excluded_columns=exclude_columns_combined
)

silver_truth["needs_one_hot_encoding"] = silver_one_hot_encoding_truths["needs_one_hot_encoding"]
silver_truth["one_hot_encoding_columns"] = silver_one_hot_encoding_truths["one_hot_encoding_columns"]

----

## Mid-Workflow Structural Review

Before I move into missingness-based feature quarantine, I take another quick look at the dataframe structure.

At this stage I have already:
- cleaned obvious junk columns
- resolved dataset identity
- added required metadata
- created canonical fields
- built anomaly-related columns
- identified an initial feature set

So this checkpoint helps me confirm the dataframe still looks structurally consistent before I make any feature-removal decisions.

In [139]:
display(dataframe.columns)

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id', 'meta__record_id', 'meta__truth_hash',
       'meta__parent_truth_hash', 'meta__pipeline_mode', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09',
       'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23',
       'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37',
       'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51',
       'machine_status', 'event_time', 'event_step',

#### Review dataframe structure and dtypes

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [140]:
display(dataframe.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220320 entries, 0 to 220319
Data columns (total 75 columns):
 #   Column                   Non-Null Count   Dtype              
---  ------                   --------------   -----              
 0   meta__dataset            220320 non-null  category           
 1   meta__split              220320 non-null  category           
 2   meta__run_id             220320 non-null  object             
 3   meta__asset_id           220320 non-null  object             
 4   meta__ingested_at_utc    220320 non-null  datetime64[us, UTC]
 5   meta__source_file        220320 non-null  string             
 6   meta__source_row_id      220320 non-null  int64              
 7   meta__record_id          220320 non-null  uint64             
 8   meta__truth_hash         220320 non-null  object             
 9   meta__parent_truth_hash  0 non-null       object             
 10  meta__pipeline_mode      220320 non-null  object             
 11  timestamp    

None

#### Review intermediate output

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [141]:
display(FEATURE_COLUMNS)

['sensor_00',
 'sensor_01',
 'sensor_02',
 'sensor_03',
 'sensor_04',
 'sensor_05',
 'sensor_06',
 'sensor_07',
 'sensor_08',
 'sensor_09',
 'sensor_10',
 'sensor_11',
 'sensor_12',
 'sensor_13',
 'sensor_14',
 'sensor_15',
 'sensor_16',
 'sensor_17',
 'sensor_18',
 'sensor_19',
 'sensor_20',
 'sensor_21',
 'sensor_22',
 'sensor_23',
 'sensor_24',
 'sensor_25',
 'sensor_26',
 'sensor_27',
 'sensor_28',
 'sensor_29',
 'sensor_30',
 'sensor_31',
 'sensor_32',
 'sensor_33',
 'sensor_34',
 'sensor_35',
 'sensor_36',
 'sensor_37',
 'sensor_38',
 'sensor_39',
 'sensor_40',
 'sensor_41',
 'sensor_42',
 'sensor_43',
 'sensor_44',
 'sensor_45',
 'sensor_46',
 'sensor_47',
 'sensor_48',
 'sensor_49',
 'sensor_50',
 'sensor_51']

#### Mid-Workflow Structural Review

This cell performs the next transformation in `Mid-Workflow Structural Review`. Review the output or logged message before relying on this result downstream.

In [142]:
dataframe_backup = dataframe.copy()
FEATURE_COLUMNS_backup = FEATURE_COLUMNS


#### Review dataframe structure and dtypes

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [143]:
#display(dataframe_backup.info())

#### Mid-Workflow Structural Review

This cell performs the next transformation in `Mid-Workflow Structural Review`. Review the output or logged message before relying on this result downstream.

In [144]:
dataframe = dataframe_backup.copy() 

#### Review dataframe structure and dtypes

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [145]:
display(dataframe.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220320 entries, 0 to 220319
Data columns (total 75 columns):
 #   Column                   Non-Null Count   Dtype              
---  ------                   --------------   -----              
 0   meta__dataset            220320 non-null  category           
 1   meta__split              220320 non-null  category           
 2   meta__run_id             220320 non-null  object             
 3   meta__asset_id           220320 non-null  object             
 4   meta__ingested_at_utc    220320 non-null  datetime64[us, UTC]
 5   meta__source_file        220320 non-null  string             
 6   meta__source_row_id      220320 non-null  int64              
 7   meta__record_id          220320 non-null  uint64             
 8   meta__truth_hash         220320 non-null  object             
 9   meta__parent_truth_hash  0 non-null       object             
 10  meta__pipeline_mode      220320 non-null  object             
 11  timestamp    

None

----

## Evaluate Missingness and Quarantine High-Missing Features

This is one of the main Silver Pre-EDA cleaning decisions.

Here I calculate missingness across the selected feature columns and apply a quarantine rule to drop features whose missingness is too high. I also keep an audit record of:
- which features were dropped
- their missingness percentages
- why they were dropped
- where the dropped-feature parquet was saved

The purpose is to reduce weak or unstable inputs before deeper EDA and modeling work.

In [146]:
def compute_missingness_percentage(
    dataframe: pd.DataFrame,
    *,
    columns: List[str],
    sort_desc: bool = True,
) -> pd.Series:
    """
    Returns a Series indexed by column name with percent missing (0..100).
    Only includes columns that exist in the dataframe.
    """
    columns = [column for column in columns if column in dataframe.columns]
    if not columns:
        return pd.Series(dtype="float64")

    missing_percentage = dataframe[columns].isna().mean().mul(100.0)
    return missing_percentage.sort_values(ascending=not sort_desc)


def quarantine_features_by_missingness(
    dataframe: pd.DataFrame,
    *,
    feature_columns: List[str],
    threshold_percentage: float,
    drop_all_null: bool = True,
    numeric_only: bool = True,
    save_dropped_dataframe: bool = False,
    dropped_dataframe_out_path: Optional["Path"] = None,
) -> Tuple[pd.DataFrame, List[str], List[str], pd.Series, Dict[str, object], Optional[pd.DataFrame]]:
    """
    Drops features whose missingness >= threshold_percentage (and optionally all-null).
    Returns:
      (clean_df, kept_features, dropped_features, missing_percentage_series, audit_dict, dropped_df_or_none)
    """
    dataframe = dataframe.copy()

    present = [column for column in feature_columns if column in dataframe.columns]
    if numeric_only:
        present = [column for column in present if pd.api.types.is_numeric_dtype(dataframe[column])]

    missing_percentage = compute_missingness_percentage(dataframe, columns=present, sort_desc=True)

    if missing_percentage.empty:
        audit = {
            "threshold_percentage": float(threshold_percentage),
            "kept_features": present,
            "dropped_features": [],
            "dropped_missing_percentage": {},
            "drop_reasons": {},
        }
        return dataframe, present, [], missing_percentage, audit, None

    over_thresh = missing_percentage >= float(threshold_percentage)
    all_null = missing_percentage >= 100.0

    mask = over_thresh
    if drop_all_null:
        mask = mask | all_null

    dropped = missing_percentage[mask].index.tolist()
    kept = [column for column in present if column not in dropped]

    drop_reasons: Dict[str, str] = {}
    dropped_missing_percentage: Dict[str, float] = {}
    for column in dropped:
        percentage = float(missing_percentage.loc[column])
        dropped_missing_percentage[column] = percentage
        drop_reasons[column] = "all_null" if percentage >= 100.0 else "missing_rate_above_threshold"

    dropped_dataframe = None
    if save_dropped_dataframe and dropped:
        dropped_dataframe = dataframe[[ "meta__record_id", *dropped ]].copy()
        #dropped_dataframe = dataframe[dropped].copy()
        if dropped_dataframe_out_path is not None:
            dropped_dataframe_out_path.parent.mkdir(parents=True, exist_ok=True)
            dropped_dataframe.to_parquet(dropped_dataframe_out_path, index=False)

    if dropped:
        dataframe = dataframe.drop(columns=dropped, errors="ignore")

    audit = {
        "threshold_percentage": float(threshold_percentage),
        "kept_features": kept,
        "dropped_features": dropped,
        "dropped_missing_percentage": dropped_missing_percentage,
        "drop_reasons": drop_reasons,
    }

    return dataframe, kept, dropped, missing_percentage, audit, dropped_dataframe

#### Quarantine features with excessive missingness

This cell performs the next transformation in `Evaluate Missingness and Quarantine High-Missing Features`. Review the output or logged message before relying on this result downstream.

In [147]:
dataframe, FEATURE_COLUMNS, dropped_features, missing_pct, missing_audit, dropped_dataframe = quarantine_features_by_missingness(
    dataframe,
    feature_columns=FEATURE_COLUMNS,
    threshold_percentage=QUARANTINE_MISSING_PCT,
    drop_all_null=True,
    numeric_only=True,
    save_dropped_dataframe=True,
    dropped_dataframe_out_path=DROPPED_SENSORS_DATA_PATH,
)


silver_truth= update_truth_section(
    silver_truth,
    "runtime_facts",
    {
        "sensor_drop_audit": {
            "threshold_pct": float(QUARANTINE_MISSING_PCT),
            "kept_features": FEATURE_COLUMNS,
            "dropped_features": missing_audit["dropped_features"],
            "dropped_missing_pct": missing_audit["dropped_missing_percentage"],
            "drop_reasons": missing_audit["drop_reasons"],
        },
    },
)



#### Review dataframe structure and dtypes

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [148]:
#display(dropped_dataframe.info())

#### Review intermediate output

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [149]:
display(missing_audit["dropped_features"])

['sensor_15', 'sensor_50']

#### Review intermediate output

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [150]:
display(missing_audit)

{'threshold_percentage': 30.0,
 'kept_features': ['sensor_00',
  'sensor_01',
  'sensor_02',
  'sensor_03',
  'sensor_04',
  'sensor_05',
  'sensor_06',
  'sensor_07',
  'sensor_08',
  'sensor_09',
  'sensor_10',
  'sensor_11',
  'sensor_12',
  'sensor_13',
  'sensor_14',
  'sensor_16',
  'sensor_17',
  'sensor_18',
  'sensor_19',
  'sensor_20',
  'sensor_21',
  'sensor_22',
  'sensor_23',
  'sensor_24',
  'sensor_25',
  'sensor_26',
  'sensor_27',
  'sensor_28',
  'sensor_29',
  'sensor_30',
  'sensor_31',
  'sensor_32',
  'sensor_33',
  'sensor_34',
  'sensor_35',
  'sensor_36',
  'sensor_37',
  'sensor_38',
  'sensor_39',
  'sensor_40',
  'sensor_41',
  'sensor_42',
  'sensor_43',
  'sensor_44',
  'sensor_45',
  'sensor_46',
  'sensor_47',
  'sensor_48',
  'sensor_49',
  'sensor_51'],
 'dropped_features': ['sensor_15', 'sensor_50'],
 'dropped_missing_percentage': {'sensor_15': 100.0,
  'sensor_50': 34.95688090050835},
 'drop_reasons': {'sensor_15': 'all_null',
  'sensor_50': 'missin

----

In [151]:
def compute_global_missingness(
    dataframe: pd.DataFrame,
    *,
    feature_columns: List[str],
    silver_truth: Dict[str, Any],
    state_list: Optional[List[str]] = None, 
    state_column_fallback: str = "machine_status",
    state_map: Optional[Dict[str, str]] = None,
    missing_state_difference_gate_percentage: float = MISSING_STATE_DIFF_GATE_PCT,
    minimum_rows_per_state_for_gate: int = MIN_ROWS_PER_STATE_FOR_GATE, 
    synthetic_suffix: str = MISSING_SYNTHETIC_SUFFIX, 
) -> Dict[str, object]: 
    
    if state_list is None:
        state_list = ["normal", "abnormal", "recovery"]

    if state_map is None:
        state_map = {
            "normal": "normal",
            "broken": "abnormal",
            "recovering": "recovery",
        }

    
    runtime_facts = cast(
        Dict[str, Any],
        silver_truth.get("runtime_facts", {}) or {},
    )

    label_resolution = cast(
        Dict[str, Any],
        runtime_facts.get("label_resolution", {}) or {},
    )

    truth_state_column = label_resolution.get("label_source_column")

    state_column = str(truth_state_column).strip() if truth_state_column and str(truth_state_column).strip() else state_column_fallback

    if state_column not in dataframe.columns:
        raise KeyError(f"Resolved state_column = '{state_column}' not found in dataframe columns.")
    
    state_column_synthetic = f"{state_column}{synthetic_suffix}"

    dataframe_copy = dataframe.copy()

    dataframe_copy[state_column_synthetic] = dataframe_copy[state_column].astype(str).str.strip().str.lower().map(state_map)
    unmapped_dataframe = dataframe_copy[state_column_synthetic].isna()
    dataframe_copy.loc[unmapped_dataframe, state_column_synthetic] = dataframe_copy.loc[unmapped_dataframe, state_column].astype(str).str.strip().str.lower()

    features = [column for column in feature_columns if column in dataframe.columns]

    if features:
        missing_global = dataframe_copy[features].isna().mean().mul(100.0)
        missing_global_percent = {str(key):float(value) for key, value in missing_global.to_dict().items()}
    else:
        missing_global_percent = {}

    rows_by_state: Dict[str, int] = {}
    missing_by_state_percentage: Dict[str, Dict[str, float]] = {}

    for state in state_list:
        dataframe_state = dataframe_copy.loc[dataframe_copy[state_column_synthetic].astype(str).eq(state)]
        rows_by_state[state] = int(len(dataframe_state))

        if not features:
            missing_by_state_percentage[state] = {}
            continue

        if len(dataframe_state) == 0:
            missing_by_state_percentage[state] = {str(feature): float("nan") for feature in features}
            continue

        missing_state = dataframe_state[features].isna().mean().mul(100.0)
        missing_by_state_percentage[state] = {str(key): float(value) for key, value in missing_state.to_dict().items()}

    # State-dependent gate
    state_dependent_flag: Dict[str, bool] = {}
    any_small_state = any(rows_by_state.get(state, 0) < minimum_rows_per_state_for_gate for state in state_list)

    for feature in features:
        if any_small_state:
            state_dependent_flag[str(feature)] = False
            continue

        values: List[float] = []
        for state in state_list:
            value = (missing_by_state_percentage.get(state, {}) or {}).get(feature)
            if value is None or not np.isfinite(value):
                continue
            values.append(float(value))

        if len(values) < 2:
            state_dependent_flag[str(feature)] = False
            continue

        spread = max(values) - min(values)
        state_dependent_flag[str(feature)] = bool(spread >= float(missing_state_difference_gate_percentage))

    missingness_audit: Dict[str, object] = {
        "missingness_pct_all": missing_global_percent,
        "missingness_pct_by_state": missing_by_state_percentage,
        "missingness_state_dependent_flag": state_dependent_flag,
        "missingness_state_gate_params": {
            "diff_gate_pct": float(missing_state_difference_gate_percentage),
            "min_rows_per_state_for_gate": int(minimum_rows_per_state_for_gate),
            "rows_by_state": {str(k): int(v) for k, v in rows_by_state.items()},
            "state_list": list(state_list),
            "state_col_resolved": state_column,
            "state_col_synth": state_column_synthetic,
            "state_map": {
                str(key): str(value)
                for key, value in (state_map or {}).items()
            },
        },
    }

    return missingness_audit


#### Summarize global missingness after quarantine

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [152]:
missingness_audit = compute_global_missingness(
    dataframe,
    feature_columns=FEATURE_COLUMNS,
    silver_truth=silver_truth,
    state_list=["normal", "abnormal", "recovery"],
    state_column_fallback="machine_status",
    state_map={
        "normal": "normal",
        "broken": "abnormal",
        "recovering": "recovery",
    },
    missing_state_difference_gate_percentage=MISSING_STATE_DIFF_GATE_PCT,
    minimum_rows_per_state_for_gate=MIN_ROWS_PER_STATE_FOR_GATE,
    synthetic_suffix=MISSING_SYNTHETIC_SUFFIX,
)

missingness_audit_map = cast(Dict[str, Any], missingness_audit)

gate = cast(
    Dict[str, Any],
    missingness_audit_map.get("missingness_state_gate_params", {}) or {},
)

missing_audit_map = cast(Dict[str, Any], missing_audit)

drop_reasons_map = cast(
    Dict[str, Any],
    missing_audit_map.get("drop_reasons", {}) or {},
)

dropped_missing_pct_map = cast(
    Dict[str, Any],
    missing_audit_map.get("dropped_missing_percentage", {}) or {},
)

rows_by_state_map = cast(
    Dict[str, Any],
    gate.get("rows_by_state", {}) or {},
)

state_map_from_gate = cast(
    Dict[str, Any],
    gate.get("state_map", {}) or {},
)

state_list_raw = gate.get("state_list", [])

if isinstance(state_list_raw, list):
    state_list_values = [str(value) for value in state_list_raw]
else:
    state_list_values = []

# Building one payload and reuse it for the Truth store and Ledger file
payload = {
    # --- quarantine decision ---
    "threshold_pct": float(QUARANTINE_MISSING_PCT),
    "dropped_count": int(len(dropped_features)),
    "dropped_features": list(dropped_features),
    "drop_reasons": dict(drop_reasons_map),
    "dropped_missing_pct": dict(dropped_missing_pct_map),
    "remaining_count": int(len(FEATURE_COLUMNS)),
    "top_missing_pct": missing_pct.head(10).to_dict(),

    # --- missingness stats for synthetic data generator ---
    "missingness_pct_all": missingness_audit_map.get("missingness_pct_all"),
    "missingness_pct_by_state": missingness_audit_map.get("missingness_pct_by_state"),
    "missingness_state_dependent_flag": missingness_audit_map.get("missingness_state_dependent_flag"),
    "missingness_state_gate_params": {
        "diff_gate_pct": gate.get("diff_gate_pct"),
        "min_rows_per_state_for_gate": gate.get("min_rows_per_state_for_gate"),
        "rows_by_state": {
            str(k): int(v)
            for k, v in rows_by_state_map.items()
        },
        "state_list": state_list_values,
        "state_col_resolved": gate.get("state_col_resolved"),
        "state_col_synth": gate.get("state_col_synth"),
        "state_map": dict(state_map_from_gate),
            },
}

# ---- Truth: runtime_facts ----
silver_truth = update_truth_section(
    silver_truth,
    "runtime_facts",
    {"missingness_quarantine": payload},
)

# ---- Truth: artifact_paths  dropped-sensors parquet ----
if DROPPED_SENSORS_DATA_PATH is not None:
    silver_truth = update_truth_section(
        silver_truth,
        "artifact_paths",
        {"silver_preeda_dropped_sensors_parquet": str(DROPPED_SENSORS_DATA_PATH)},
    )

# ---- Ledger: same payload ----
ledger.add(
    kind="decision",
    step="missingness_quarantine",
    message="Dropped features exceeding missingness threshold (or all-null).",
    why="High-missingness features add noise/instability to EDA + modeling.",
    consequence="Feature set shrinks; synthetic generator will replicate missingness for dropped sensors.",
    data=payload,
    logger=logger,
)

print("Saved missingness_quarantine payload to Truth + Ledger.")
print("Dropped:", payload["dropped_features"])

2026-06-14 22:04:17,361 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:04:17.361052+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'decision', 'step': 'missingness_quarantine', 'message': 'Dropped features exceeding missingness threshold (or all-null).', 'why': 'High-missingness features add noise/instability to EDA + modeling.', 'consequence': 'Feature set shrinks; synthetic generator will replicate missingness for dropped sensors.', 'data': {'threshold_pct': 30.0, 'dropped_count': 2, 'dropped_features': ['sensor_15', 'sensor_50'], 'drop_reasons': {'sensor_15': 'all_null', 'sensor_50': 'missing_rate_above_threshold'}, 'dropped_missing_pct': {'sensor_15': 100.0, 'sensor_50': 34.95688090050835}, 'remaining_count': 50, 'top_missing_pct': {'sensor_15': 100.0, 'sensor_50': 34.95688090050835, 'sensor_51': 6.982116920842412, 'sensor_00': 4.633260711692085, 'sensor_07': 2.474128540305011, 'sensor_08': 2.3179920116194626, 'se

Saved missingness_quarantine payload to Truth + Ledger.
Dropped: ['sensor_15', 'sensor_50']


#### Define configuration mapping guards

This cell performs the next transformation in `----`. Review the output or logged message before relying on this result downstream.

In [153]:
sensor = "sensor_48"

missingness_audit_map = cfg_require_mapping(
    missingness_audit,
    "missingness_audit",
)

missingness_pct_all = cfg_require_mapping(
    missingness_audit_map.get("missingness_pct_all", {}),
    "missingness_audit['missingness_pct_all']",
)

missingness_pct_by_state = cfg_require_mapping(
    missingness_audit_map.get("missingness_pct_by_state", {}),
    "missingness_audit['missingness_pct_by_state']",
)

normal_missingness = cfg_require_mapping(
    missingness_pct_by_state.get("normal", {}),
    "missingness_pct_by_state['normal']",
)

abnormal_missingness = cfg_require_mapping(
    missingness_pct_by_state.get("abnormal", {}),
    "missingness_pct_by_state['abnormal']",
)

recovery_missingness = cfg_require_mapping(
    missingness_pct_by_state.get("recovery", {}),
    "missingness_pct_by_state['recovery']",
)

print("selected_sensor:", sensor)
print("global:", missingness_pct_all.get(sensor))
print("normal:", normal_missingness.get(sensor))
print("abnormal:", abnormal_missingness.get(sensor))
print("recovery:", recovery_missingness.get(sensor))

selected_sensor: sensor_48
global: 0.012254901960784314
normal: 0.010688120639732603
abnormal: 0.0
recovery: 0.03453754230848933


#### Review intermediate output

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [154]:
display(silver_truth)

{'truth_version': 'truth__001',
 'dataset_name': 'pump',
 'layer_name': 'silver',
 'process_run_id': 'silver_process__20260614T220330Z',
 'pipeline_mode': 'batch',
 'parent_truth_hash': 'b90b025534b790481484fe4870e5bff95c50143212898e9ff02b88401303da69',
 'source_fingerprint': {},
 'config_snapshot': {'silver_version': 'silver__001',
  'cleaning_recipe_id': 'silver__clean__dataset__agnostic__v001',
  'dataset_name_config': 'pump',
  'dataset_name_parent_truth': 'pump',
  'run_id_default_fallback': 'synthetic_run_001',
  'quarantine_missing_pct': 30.0,
  'min_time_parse_success_percent': 95.0,
  'min_step_parse_success_percent': 95.0,
  'pipeline_mode': 'batch'},
 'runtime_facts': {'parent_layer_name': 'bronze',
  'parent_truth_hash': 'b90b025534b790481484fe4870e5bff95c50143212898e9ff02b88401303da69',
  'dataset_name_from_parent_truth': 'pump',
  'dataset_validation': {'dataset_name': 'pump',
   'dataset_source_column': 'meta__dataset',
   'dataset_method': 'unique'},
  'processed_at_utc

### Ask

Why am I treating missingness as both a cleaning issue and an analysis issue here?

### Answer

Missingness is not only a technical data-quality problem. It can also carry real pattern information.

In this notebook I look at missingness in two ways:
1. **quarantine logic** for deciding whether a feature is too incomplete to keep
2. **state-aware missingness tracking** so I can preserve useful missingness behavior for the synthetic data generator later

So the goal is not just to drop bad columns blindly. The goal is to remove features that are too weak to keep while still recording missingness patterns that may matter later.

----

## Finalize the Feature Lists After Quarantine

Once the missingness quarantine step is complete, I update the feature-related objects so they reflect the final kept schema instead of the pre-quarantine version.

That includes:
- final `FEATURE_COLUMNS`
- updated `FEATURE_GROUPS`
- refreshed `FEATURE_INFO`
- a compact summary of the missingness quarantine decision

This matters because I do not want downstream outputs referencing columns that were already removed.

In [155]:
# Normalize the dropped features just in case. 
dropped_features = dropped_features or []
dropped_set = set(dropped_features)

# 1) Ensure FEATURE_COLUMNS is "final schema" (already updated by quarantine return)
FEATURE_COLUMNS = [column for column in FEATURE_COLUMNS if column not in dropped_set]

# 2) Rebuild FEATURE_GROUPS to only include final schema columns
#    - keeps original group names
#    - drops columns that were quarantined
#    - drops columns not in FEATURE_COLUMNS 
if "FEATURE_GROUPS" in globals() and isinstance(FEATURE_GROUPS, dict):
    FEATURE_GROUPS = {
        group: [column for column in columns if column in FEATURE_COLUMNS and column not in dropped_set]
        for group, columns in FEATURE_GROUPS.items()
    }
    # remove empty groups
    FEATURE_GROUPS = {group: columns for group, columns in FEATURE_GROUPS.items() if columns}
else:
    FEATURE_GROUPS = {}

# 3) Update FEATURE_INFO counts and quarantine summary
if "FEATURE_INFO" not in globals() or not isinstance(FEATURE_INFO, dict):
    FEATURE_INFO = {}

FEATURE_INFO["selected_feature_count"] = int(len(FEATURE_COLUMNS))
FEATURE_INFO["group_counts"] = {group: int(len(columns)) for group, columns in FEATURE_GROUPS.items()}
FEATURE_INFO["missingness_quarantine"] = {
    "threshold_pct": float(QUARANTINE_MISSING_PCT),
    "dropped_count": int(len(dropped_features)),
    "dropped_features": list(dropped_features),
}

# Attach my audit here
if "missing_audit" in globals() and isinstance(missing_audit, dict):
    FEATURE_INFO["missingness_quarantine"].update(
        {
            "dropped_missing_pct": missing_audit.get("dropped_missing_percentage", {}),
            "drop_reasons": missing_audit.get("drop_reasons", {}),
        }
    )

# 4) Quick sanity check
print("Final FEATURE_COLUMNS:", len(FEATURE_COLUMNS))
print("Final FEATURE_GROUPS:", len(FEATURE_GROUPS), "groups")
print("Dropped:", FEATURE_INFO["missingness_quarantine"]["dropped_count"], FEATURE_INFO["missingness_quarantine"]["dropped_features"])

Final FEATURE_COLUMNS: 50
Final FEATURE_GROUPS: 1 groups
Dropped: 2 ['sensor_15', 'sensor_50']


----

## Create a Stable Feature Set Identifier

Now that the final feature set is known, I create a deterministic feature-set identifier based on the selected columns.

This gives me a stable way to refer to the exact feature schema used in this Silver output. If the feature list changes later, the identifier changes too, which makes comparisons and tracking more reliable.

In [156]:
def build_feature_set_identifier(feature_columns: List[str]) -> str:
    """
    Deterministic identifier for a feature set based on sorted column names.
    """
    normalized = "|".join(sorted([str(name) for name in feature_columns]))
    digest = hashlib.md5(normalized.encode("utf-8")).hexdigest()
    return f"feature_set__{digest}"

#### Apply feature exclusion rules

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [157]:
FEATURE_SET_IDENTIFIER = build_feature_set_identifier(FEATURE_COLUMNS)
FEATURE_COUNT = int(len(FEATURE_COLUMNS))

silver_truth = update_truth_section(
    silver_truth,
    "runtime_facts",
    {
        "feature_set": {
            "feature_set_id": FEATURE_SET_IDENTIFIER,
            "feature_count": FEATURE_COUNT,
        }
    },
)

ledger.add(
    kind="decision",
    step="finalize_feature_set",
    message="Finalized feature set and wrote feature set metadata to Truth Store.",
    data={
        "feature_set_id": FEATURE_SET_IDENTIFIER,
        "label_source_column": LABEL_SOURCE_COLUMN,
        "label_source_type": LABEL_SOURCE_TYPE,
        "exclude_prefixes": list(DEFAULT_EXCLUDE_PREFIXES),
        "exclude_columns": list(exclude_columns_combined),
        "feature_count": FEATURE_COUNT,
        "selected_feature_columns": list(FEATURE_COLUMNS),
        "feature_groups": {g: list(cols) for g, cols in FEATURE_GROUPS.items()},
        "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
    },
    logger=logger,
)

2026-06-14 22:04:19,258 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:04:19.258763+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'decision', 'step': 'finalize_feature_set', 'message': 'Finalized feature set and wrote feature set metadata to Truth Store.', 'why': None, 'consequence': None, 'data': {'feature_set_id': 'feature_set__bea5fdd737ae53fcd80ca84cafcd0c40', 'label_source_column': 'machine_status', 'label_source_type': 'status', 'exclude_prefixes': ['meta__', 'raw__'], 'exclude_columns': ['event_time', 'event_step', 'time_index', 'event_date', 'meta__event_id', 'anomaly_flag', 'is_anomaly', 'is_normal', 'status_normal_value'], 'feature_count': 50, 'selected_feature_columns': ['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_16', 'sensor_17', 'sensor_18', 'sens

{'ts_utc': '2026-06-14T22:04:19.258763+00:00',
 'stage': 'silver_preeda',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'decision',
 'step': 'finalize_feature_set',
 'message': 'Finalized feature set and wrote feature set metadata to Truth Store.',
 'why': None,
 'consequence': None,
 'data': {'feature_set_id': 'feature_set__bea5fdd737ae53fcd80ca84cafcd0c40',
  'label_source_column': 'machine_status',
  'label_source_type': 'status',
  'exclude_prefixes': ['meta__', 'raw__'],
  'exclude_columns': ['event_time',
   'event_step',
   'time_index',
   'event_date',
   'meta__event_id',
   'anomaly_flag',
   'is_anomaly',
   'is_normal',
   'status_normal_value'],
  'feature_count': 50,
  'selected_feature_columns': ['sensor_00',
   'sensor_01',
   'sensor_02',
   'sensor_03',
   'sensor_04',
   'sensor_05',
   'sensor_06',
   'sensor_07',
   'sensor_08',
   'sensor_09',
   'sensor_10',
   'sensor_11',
   'sensor_12',
   'sensor_13',
   'sensor_14',
   'sensor_16',
   'senso

----

## Reorder the Silver Columns into a Cleaner Final Layout

Here I reorganize the dataframe columns into a more intentional order.

The idea is to place the Silver output in a layout that is easier to inspect and easier to pass downstream. In general, I want:
- metadata columns first
- canonical identity and ordering columns next
- label-related columns after that
- selected feature columns after that
- any remaining columns at the end

This does not change the underlying values. It just makes the final Silver dataframe cleaner and more consistent.

In [158]:
def safe_list_columns(columns: List[str], existing_columns: List[str]) -> List[str]:
    kept: List[str] = []
    for column in columns:
        if column in existing_columns:
            kept.append(column)
    return kept


#### Define metadata column ordering helper

This cell defines helper logic used by the surrounding `Reorder the Silver Columns into a Cleaner Final Layout` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [159]:

def collect_meta_columns(existing_columns: List[str]) -> List[str]:
    meta_columns: List[str] = []
    for column in existing_columns:
        if column.startswith("meta__"):
            meta_columns.append(column)
    return meta_columns


#### Define metadata column ordering helper

This cell defines helper logic used by the surrounding `Reorder the Silver Columns into a Cleaner Final Layout` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [160]:

def reorder_silver_columns(
    dataframe: pd.DataFrame,
    *,
    canonical_non_meta_order: List[str],
    label_columns_order: List[str],
    feature_columns: List[str],
) -> pd.DataFrame:
    existing_columns = list(dataframe.columns)

    meta_columns = collect_meta_columns(existing_columns)
    meta_columns = sorted(meta_columns)

    canonical_columns = safe_list_columns(canonical_non_meta_order, existing_columns)
    label_columns = safe_list_columns(label_columns_order, existing_columns)

    feature_columns_present = safe_list_columns(feature_columns, existing_columns)

    # Remainder columns (preserve original order for anything not in the primary groups)
    ordered_set = set(meta_columns) | set(canonical_columns) | set(label_columns) | set(feature_columns_present)

    remainder_columns: List[str] = []
    for column in existing_columns:
        if column not in ordered_set:
            remainder_columns.append(column)

    final_order: List[str] = []
    final_order.extend(meta_columns)
    final_order.extend(canonical_columns)
    final_order.extend(label_columns)
    final_order.extend(feature_columns_present)
    final_order.extend(remainder_columns)

    return dataframe[final_order].copy()

#### Review current column layout

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [161]:
display(dataframe.columns)

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id', 'meta__record_id', 'meta__truth_hash',
       'meta__parent_truth_hash', 'meta__pipeline_mode', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09',
       'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24',
       'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38',
       'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_51', 'machine_status', 'event_time',
       'event_step', 'time_index', 'meta__even

#### Define final Silver column ordering

This cell performs the next transformation in `Reorder the Silver Columns into a Cleaner Final Layout`. Review the output or logged message before relying on this result downstream.

In [162]:
dataframe = reorder_silver_columns(
    dataframe,
    canonical_non_meta_order=CANONICAL_NON_META_ORDER,
    label_columns_order=LABEL_COLUMNS_ORDER,
    feature_columns=FEATURE_COLUMNS,
)

----

In [163]:
display(dataframe.columns)

Index(['meta__asset_id', 'meta__dataset', 'meta__episode_id', 'meta__event_id', 'meta__ingested_at_utc', 'meta__parent_truth_hash', 'meta__pipeline_mode', 'meta__record_id', 'meta__run_id',
       'meta__source_file', 'meta__source_row_id', 'meta__split', 'meta__truth_hash', 'event_time', 'event_step', 'time_index', 'event_date', 'anomaly_flag', 'is_anomaly', 'is_normal',
       'status_normal_value', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12',
       'sensor_13', 'sensor_14', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27',
       'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41',
       'sensor_42', 'sensor_43', 'sensor_44', 's

----

## Run Final Quick Quality Checks

Before saving the Silver output, I run a set of quick quality checks across the dataframe.

These checks help summarize:
- total rows and columns
- duplicate rows
- duplicate event IDs
- top missingness among numeric features
- anomaly counts and anomaly rate

This gives me one more compact validation step before I stamp truth metadata and export the final artifacts.

In [164]:
def compute_quick_quality_checks(
    dataframe: pd.DataFrame,
    *,
    feature_columns: List[str],
    event_id_column: str = "meta__event_id",
    anomaly_flag_column: str = "anomaly_flag",
) -> Dict[str, Any]:
    total_rows = int(len(dataframe))

    # Duplicate checks
    duplicate_row_count = int(dataframe.duplicated().sum())

    duplicate_event_id_count = None
    if event_id_column in dataframe.columns:
        duplicate_event_id_count = int(dataframe[event_id_column].duplicated().sum())

    # Missingness for feature columns (limit logging size)
    numeric_missingness: Dict[str, float] = {}
    numeric_feature_count = 0

    for column in feature_columns:
        if column not in dataframe.columns:
            continue
        if pd.api.types.is_numeric_dtype(dataframe[column]):
            numeric_feature_count += 1
            missing_percent = float(dataframe[column].isna().mean() * 100.0) if total_rows > 0 else 0.0
            numeric_missingness[column] = missing_percent

    # Keep only top 25 missingness entries (largest first) to avoid huge ledger payload
    sorted_missingness = sorted(numeric_missingness.items(), key=lambda item: item[1], reverse=True)
    top_missingness = dict(sorted_missingness[:25])

    # Anomaly rate
    anomaly_rate_percent = None
    anomaly_counts = None
    if anomaly_flag_column in dataframe.columns and total_rows > 0:
        anomaly_rate_percent = float(dataframe[anomaly_flag_column].mean() * 100.0)
        value_counts = dataframe[anomaly_flag_column].value_counts(dropna=False).to_dict()
        anomaly_counts = {str(key): int(value) for key, value in value_counts.items()}

    return {
        "total_rows": total_rows,
        "total_columns": int(len(dataframe.columns)),
        "duplicate_row_count": duplicate_row_count,
        "duplicate_event_id_count": duplicate_event_id_count,
        "numeric_feature_count": int(numeric_feature_count),
        "top_numeric_missingness_percent": top_missingness,
        "anomaly_rate_percent": anomaly_rate_percent,
        "anomaly_counts": anomaly_counts,
    }

#### Run final Silver quality checks

This cell performs the next transformation in `Run Final Quick Quality Checks`. Review the output or logged message before relying on this result downstream.

In [165]:
quality_info = compute_quick_quality_checks(
    dataframe,
    feature_columns=FEATURE_COLUMNS,
)

### Ask

What does this final quality summary help me decide?

### Answer

This summary helps answer whether the dataframe is ready to be treated as a cleaned Silver Pre-EDA output.

At this point I want to see that:
- duplicate issues are either absent or understood
- the feature set looks stable after quarantine
- missingness is documented
- anomaly labeling looks reasonable at a high level
- the dataframe is in a condition that can be safely saved and reused

So this is less about finding deep insights and more about confirming output readiness.

----

## Build the Silver Feature Registry

This step creates the feature registry artifact for the Silver output.

The feature registry acts like a structured summary of the final feature schema. It records things like:
- dataset identity
- row and column counts
- feature set ID
- final selected features
- feature groups
- exclusion logic
- label source details
- pipeline context

I want this saved as a separate artifact because it makes the final Silver schema easier to inspect without reopening the whole notebook.

In [166]:
feature_registry: Dict[str, Any] = {
    "dataset_name": DATASET_NAME,
    "row_count": int(len(dataframe)),
    "column_count": int(len(dataframe.columns)),
    "feature_set_id": FEATURE_SET_IDENTIFIER,
    "feature_count": int(len(FEATURE_COLUMNS)),
    "feature_columns": list(FEATURE_COLUMNS),
    "feature_groups": {
        group_name: list(columns) for group_name, columns in FEATURE_GROUPS.items()
    },
    "feature_info": FEATURE_INFO,
    "exclude_prefixes": list(DEFAULT_EXCLUDE_PREFIXES),
    "exclude_columns": list(exclude_columns_combined),
    "label_source_column": LABEL_SOURCE_COLUMN,
    "label_source_type": LABEL_SOURCE_TYPE,
    "quarantine_missing_pct": float(QUARANTINE_MISSING_PCT),
    "pipeline_mode": PIPELINE_MODE,
    "process_run_id": SILVER_PROCESS_RUN_ID,
}

----

In [167]:
if not dataframe.columns.is_unique:
    duplicates_found = dataframe.columns[dataframe.columns.duplicated()].tolist()

    raise ValueError(f"Duplicate columns detected before save: {duplicates_found}")

----

In [168]:
display(dataframe.columns)

Index(['meta__asset_id', 'meta__dataset', 'meta__episode_id', 'meta__event_id', 'meta__ingested_at_utc', 'meta__parent_truth_hash', 'meta__pipeline_mode', 'meta__record_id', 'meta__run_id',
       'meta__source_file', 'meta__source_row_id', 'meta__split', 'meta__truth_hash', 'event_time', 'event_step', 'time_index', 'event_date', 'anomaly_flag', 'is_anomaly', 'is_normal',
       'status_normal_value', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12',
       'sensor_13', 'sensor_14', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27',
       'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41',
       'sensor_42', 'sensor_43', 'sensor_44', 's

#### Preview current dataframe rows

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [169]:
display(dataframe.head())

,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status
0,asset__001,pump,0,pump:asset__001:run__001:0,2026-06-14 20:37:14.939771+00:00,None,batch,14598431322315673869,run__001,sensor.csv,0,unsplit,b90b025534b790481484fe4870e5bff95c50143212898e...,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,201.3889,2018-04-01 00:00:00,NORMAL
1,asset__001,pump,0,pump:asset__001:run__001:1,2026-06-14 20:37:14.939771+00:00,None,batch,15954729095895098000,run__001,sensor.csv,1,unsplit,b90b025534b790481484fe4870e5bff95c50143212898e...,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,201.3889,2018-04-01 00:01:00,NORMAL
2,asset__001,pump,0,pump:asset__001:run__001:2,2026-06-14 20:37:14.939771+00:00,None,batch,10041703297090838359,run__001,sensor.csv,2,unsplit,b90b025534b790481484fe4870e5bff95c50143212898e...,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.444734,47.35243,53.2118,46.397570,638.8889,73.54598,13.32465,16.03733,15.61777,15.01013,37.86777,48.17723,32.08894,1.708474,420.8480,462.7798,459.6364,2.500062,666.2234,399.9418,880.4237,501.3617,982.7342,631.1326,740.8031,849.8997,454.2390,778.5734,715.6266,661.5740,721.8750,694.7721,441.2635,169.9820,343.1955,200.9694,93.90508,41.40625,31.25000,69.53125,30.46875,31.770830,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,203.7037,2018-04-01 00:02:00,NORMAL
3,asset__001,pump,0,pump:asset__001:run__001:3,2026-06-14 20:37:14.939771+00:00,None,batch,2072036352569063261,run__001,sensor.csv,3,unsplit,b90b025534b790481484fe4870e5bff95c50143212898e...,2018-04-01 00:03:00+00:00,3,3,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.460474,47.09201,53.1684,46.397568,628.1250,76.98898,13.31742,16.24711,15.69734,15.08247,38.57977,48.65607,31.67221,1.579427,420.7494,462.8980,460.8858,2.509521,666.0114,399.1046,878.8917,499.0430,977.7520,625.4076,739.2722,847.7579,474.8731,779.5091,690.4011,686.1111,754.6875,683.3831,446.2493,166.4987,343.9586,193.1689,101.04060,41.92708,31.51042,72.13541,30.46875,31.510420,40.88541,39.062500,64.81481,51.21528,38.194440,155.9606,66.84028,203.1250,2018-04-01 00:03:00,NORMAL
4,asset__001,pump,0,pump:asset__001:run__001:4,2026-06-14 20:37:14.939771+00:00,None,batch,4365040424004714369,run__001,sensor.csv,4,unsplit,b90b025534b790481484fe4870e5bff95c50143212898e..

----

## Build the Silver Truth Record and Save the Final Outputs

Now I am at the stage where the Silver dataframe becomes a formal pipeline artifact.

This part of the notebook does several important things:
- updates the truth record with source and runtime details
- builds the final Silver truth record
- stamps lineage columns onto the dataframe
- saves the truth file
- saves the final Silver parquet
- saves the feature registry

This is the point where the notebook output moves from "working dataframe" to "tracked deliverable."

In [170]:
#SILVER_ARTIFACTS_PATH = paths.artifacts / "silver" / DATASET_NAME
#SILVER_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

#SILVER_TRAIN_DATA_FILE_NAME = f"{DATASET_NAME}__silver__train.parquet"

silver_truth = update_truth_section(
    silver_truth,
    "source_fingerprint",
    build_file_fingerprint(bronze_data_path),
)

silver_truth = update_truth_section(
    silver_truth,
    "runtime_facts",
    {
        "source_run_ids": (
            dataframe["meta__run_id"].dropna().astype(str).unique().tolist()
            if "meta__run_id" in dataframe.columns
            else []
        ),
        "quality_info": quality_info if "quality_info" in locals() else {},
    },
)

silver_truth = update_truth_section(
    silver_truth,
    "artifact_paths",
    {
        "bronze_source_path": str(bronze_data_path),
        "silver_output_dir": str(SILVER_TRAIN_DATA_PATH),
        "silver_output_file_name": SILVER_TRAIN_DATA_FILE_NAME,
        "feature_registry_dir": str(SILVER_ARTIFACTS_PATH),
        "silver_preeda_dropped_sensors_path": str(DROPPED_SENSORS_DATA_PATH),
    },
)

silver_truth = update_truth_section(
    silver_truth,
    "notes",
    {
        "purpose": "Silver preprocessing / canonicalization truth record",
    },
)

silver_truth_record = build_truth_record(
    truth_base=silver_truth,
    row_count=len(dataframe),
    column_count=dataframe.shape[1],
    meta_columns=identify_meta_columns(dataframe),
    feature_columns=identify_feature_columns(dataframe),
)

SILVER_TRUTH_HASH = silver_truth_record["truth_hash"]

dataframe = stamp_truth_columns(
    dataframe,
    truth_hash=SILVER_TRUTH_HASH,
    parent_truth_hash=SILVER_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

feature_registry["truth_hash"] = SILVER_TRUTH_HASH
feature_registry["parent_truth_hash"] = SILVER_PARENT_TRUTH_HASH

silver_truth_path = save_truth_record(
    silver_truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name=LAYER_NAME,
)

append_truth_index(
    silver_truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

ledger.add(
    kind="step",
    step="build_silver_truth_record",
    message="Built and saved Silver truth record and stamped only truth lineage columns to dataframe.",
    data={
        "silver_truth_hash": SILVER_TRUTH_HASH,
        "silver_parent_truth_hash": SILVER_PARENT_TRUTH_HASH,
        "silver_truth_path": str(silver_truth_path),
        "pipeline_mode": PIPELINE_MODE,
        "process_run_id": SILVER_PROCESS_RUN_ID,
    },
    logger=logger,
)

2026-06-14 22:04:25,283 | INFO | capstone.silver.preeda | LEDGER | {'ts_utc': '2026-06-14T22:04:25.283129+00:00', 'stage': 'silver_preeda', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'build_silver_truth_record', 'message': 'Built and saved Silver truth record and stamped only truth lineage columns to dataframe.', 'why': None, 'consequence': None, 'data': {'silver_truth_hash': '6d78344b915d1f5c31dac560d155433f966c3cfca60380e4ba597c2f918860a3', 'silver_parent_truth_hash': 'b90b025534b790481484fe4870e5bff95c50143212898e9ff02b88401303da69', 'silver_truth_path': '/workspace/artifacts/truths/silver/pump__silver__truth__6d78344b915d1f5c31dac560d155433f966c3cfca60380e4ba597c2f918860a3.json', 'pipeline_mode': 'batch', 'process_run_id': 'silver_process__20260614T220330Z'}}


{'ts_utc': '2026-06-14T22:04:25.283129+00:00',
 'stage': 'silver_preeda',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'step',
 'step': 'build_silver_truth_record',
 'message': 'Built and saved Silver truth record and stamped only truth lineage columns to dataframe.',
 'why': None,
 'consequence': None,
 'data': {'silver_truth_hash': '6d78344b915d1f5c31dac560d155433f966c3cfca60380e4ba597c2f918860a3',
  'silver_parent_truth_hash': 'b90b025534b790481484fe4870e5bff95c50143212898e9ff02b88401303da69',
  'silver_truth_path': '/workspace/artifacts/truths/silver/pump__silver__truth__6d78344b915d1f5c31dac560d155433f966c3cfca60380e4ba597c2f918860a3.json',
  'pipeline_mode': 'batch',
  'process_run_id': 'silver_process__20260614T220330Z'}}

#### Assign a reproducible feature set identifier

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [171]:
#SILVER_TRAIN_DATA_FILE_NAME = f"{DATASET_NAME}__silver__train.parquet"

saved_parquet_path = save_data(
    dataframe,
    file_path=SILVER_TRAIN_DATA_PATH,
    file_name=SILVER_TRAIN_DATA_FILE_NAME,
    create_dirs=True,
    index=False,
)

saved_registry_path = save_json(
    feature_registry,
    file_path=SILVER_REGISTRY_DIR,
    file_name=FEATURE_REGISTRY_FILE_NAME,
    create_dirs=True,
    indent=2,
)

ledger.add(
    kind="step",
    step="silver_finalize_export",
    message="Finalized Silver export under the strict truth-store meta contract.",
    data={
        "dataset_name": DATASET_NAME,
        "silver_parquet_path": str(saved_parquet_path),
        "feature_registry_path": str(saved_registry_path),
        "quality_info": quality_info,
        "feature_set_id": FEATURE_SET_IDENTIFIER,
        "feature_count": FEATURE_COUNT,
        "silver_truth_hash": SILVER_TRUTH_HASH,
        "silver_parent_truth_hash": SILVER_PARENT_TRUTH_HASH,
        "pipeline_mode": PIPELINE_MODE,
        "process_run_id": SILVER_PROCESS_RUN_ID,
        "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
    },
    logger=logger,
)

2026-06-14 22:04:25,631 | INFO | capstone.file_io | Saving DataFrame to Parquet: /workspace/data/silver/train/pump__silver__train.parquet
2026-06-14 22:04:28,601 | INFO | capstone.file_io | Saved: pump__silver__train.parquet | shape=(220320, 73) | columns=['meta__asset_id', 'meta__dataset', 'meta__episode_id', 'meta__event_id', 'meta__ingested_at_utc', 'meta__parent_truth_hash', 'meta__pipeline_mode', 'meta__record_id', 'meta__run_id', 'meta__source_file', 'meta__source_row_id', 'meta__split', 'meta__truth_hash', 'event_time', 'event_step', 'time_index', 'event_date', 'anomaly_flag', 'is_anomaly', 'is_normal', 'status_normal_value', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_

{'ts_utc': '2026-06-14T22:04:28.631417+00:00',
 'stage': 'silver_preeda',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'step',
 'step': 'silver_finalize_export',
 'message': 'Finalized Silver export under the strict truth-store meta contract.',
 'why': None,
 'consequence': None,
 'data': {'dataset_name': 'pump',
  'silver_parquet_path': '/workspace/data/silver/train/pump__silver__train.parquet',
  'feature_registry_path': '/workspace/artifacts/silver/pump/preeda/registry/pump__silver__feature_registry.json',
  'quality_info': {'total_rows': 220320,
   'total_columns': 73,
   'duplicate_row_count': 0,
   'duplicate_event_id_count': 0,
   'numeric_feature_count': 50,
   'top_numeric_missingness_percent': {'sensor_51': 6.982116920842412,
    'sensor_00': 4.633260711692085,
    'sensor_07': 2.474128540305011,
    'sensor_08': 2.3179920116194626,
    'sensor_06': 2.177741466957153,
    'sensor_09': 2.0856027596223674,
    'sensor_01': 0.16748366013071894,
    'sensor_30': 

----

## Save the Ledger Artifact

After the main exports are complete, I save the Silver ledger to JSON.

This gives me a structured record of the major notebook actions and decisions in one place, which helps with traceability and makes the preprocessing work easier to review later.

In [172]:
# Save the ledger
saved_ledger_path = ledger.write_json(
    SILVER_LINEAGE_DIR / f"silver__{DATASET_NAME}__ledger.json"
)


----

## Finalize Experiment Tracking

This step wraps up the W&B tracking for the Silver stage.

By this point, the dataframe and its related artifacts have already been created. So this step is mainly about closing out the run cleanly and making sure the final Silver stage metadata is properly registered.

In [173]:
finalize_info = finalize_wandb_stage(
    run=wandb_run,
    stage=STAGE,
    dataframe=dataframe,
    project_root=paths.root,
    logs_dir=paths.logs,
    dataset_dirs=[paths.data_silver_train],
    dataset_artifact_name=f"{DATASET_NAME}-{STAGE}-dataset",
    logger=logger,
    notebook_path=None,
    aliases=("latest",),
    table_key=None,
    table_n=15,
    profile=False,
)

# Close the W&B run
wandb_run.finish()

bronze_cols,▁
bronze_rows,▁
cols,▁
memory_mb,▁
rows,▁
bronze_cols,66
bronze_rows,220320
cols,73
memory_mb,269.52536
rows,220320


----

## Run a Final Lineage and Consistency Check

Before I consider the Silver Pre-EDA notebook complete, I run one final sanity check on the saved output.

This verifies that:
- required lineage columns exist
- the dataframe truth hash matches the saved truth record
- the parent truth hash is correct
- the truth file exists
- row and column counts match what was saved

I like ending with this because it confirms that the Silver output is not only saved, but also internally consistent and traceable.

In [174]:

def require_int_value(value: object, name: str) -> int:
    if value is None:
        raise ValueError(f"{name} cannot be None.")

    try:
        return int(cast(Any, value))
    except (TypeError, ValueError) as exc:
        raise TypeError(
            f"{name} must be convertible to int, "
            f"got {type(value).__name__}: {value!r}"
        ) from exc

#### Define integer validation helper

This cell runs guardrail checks before the notebook continues. These checks reduce the risk of carrying missing columns, invalid labels, or inconsistent row counts into later outputs.

In [175]:
required_silver_meta_columns = [
    "meta__truth_hash",
    "meta__parent_truth_hash",
    "meta__pipeline_mode",
]

missing_silver_meta_columns = [
    column_name
    for column_name in required_silver_meta_columns
    if column_name not in dataframe.columns
]
if missing_silver_meta_columns:
    raise ValueError(
        f"Silver dataframe is missing required lineage columns: {missing_silver_meta_columns}"
    )

silver_dataframe_truth_hash = extract_truth_hash(dataframe)
if silver_dataframe_truth_hash is None:
    raise ValueError("Silver dataframe does not contain a readable meta__truth_hash value.")

if silver_dataframe_truth_hash != SILVER_TRUTH_HASH:
    raise ValueError(
        "Silver dataframe truth hash does not match SILVER_TRUTH_HASH:\n"
        f"dataframe={silver_dataframe_truth_hash}\n"
        f"record={SILVER_TRUTH_HASH}"
    )

silver_parent_values = dataframe["meta__parent_truth_hash"].dropna().astype(str).unique().tolist()
if not silver_parent_values:
    raise ValueError("Silver dataframe is missing populated meta__parent_truth_hash values.")

if len(silver_parent_values) != 1:
    raise ValueError(
        "Silver dataframe has multiple parent truth hashes:\n"
        f"{silver_parent_values}"
    )

if silver_parent_values[0] != SILVER_PARENT_TRUTH_HASH:
    raise ValueError(
        "Silver dataframe parent truth hash does not match SILVER_PARENT_TRUTH_HASH:\n"
        f"dataframe_parent={silver_parent_values[0]}\n"
        f"silver_parent_truth={SILVER_PARENT_TRUTH_HASH}"
    )

if not Path(silver_truth_path).exists():
    raise FileNotFoundError(f"Silver truth file was not created: {silver_truth_path}")

loaded_silver_truth_raw = load_json(silver_truth_path)

if not isinstance(loaded_silver_truth_raw, dict):
    raise TypeError(
        "loaded_silver_truth must be a dictionary, "
        f"got {type(loaded_silver_truth_raw).__name__}: {loaded_silver_truth_raw!r}"
    )

loaded_silver_truth = cast(Dict[str, Any], loaded_silver_truth_raw)

loaded_silver_truth_hash = str(loaded_silver_truth.get("truth_hash", "")).strip()

if loaded_silver_truth_hash != SILVER_TRUTH_HASH:
    raise ValueError(
        "Saved Silver truth file hash does not match SILVER_TRUTH_HASH:\n"
        f"file={loaded_silver_truth_hash}\n"
        f"record={SILVER_TRUTH_HASH}"
    )

loaded_silver_parent_truth_hash = str(
    loaded_silver_truth.get("parent_truth_hash", "")
).strip()

if loaded_silver_parent_truth_hash != SILVER_PARENT_TRUTH_HASH:
    raise ValueError(
        "Saved Silver truth file parent hash does not match SILVER_PARENT_TRUTH_HASH:\n"
        f"truth={loaded_silver_parent_truth_hash}\n"
        f"silver_parent={SILVER_PARENT_TRUTH_HASH}"
    )

silver_truth_row_count = require_int_value(
    loaded_silver_truth.get("row_count"),
    "loaded_silver_truth['row_count']",
)

silver_truth_column_count = require_int_value(
    loaded_silver_truth.get("column_count"),
    "loaded_silver_truth['column_count']",
)

if silver_truth_row_count != len(dataframe):
    raise ValueError(
        "Silver truth row_count does not match dataframe row count:\n"
        f"truth={silver_truth_row_count}\n"
        f"dataframe={len(dataframe)}"
    )

if silver_truth_column_count != dataframe.shape[1]:
    raise ValueError(
        "Silver truth column_count does not match stamped dataframe column count:\n"
        f"truth={silver_truth_column_count}\n"
        f"dataframe={dataframe.shape[1]}"
    )

print("Silver PreEDA lineage sanity check passed.")

2026-06-14 22:04:39,242 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/truths/silver/pump__silver__truth__6d78344b915d1f5c31dac560d155433f966c3cfca60380e4ba597c2f918860a3.json


Silver PreEDA lineage sanity check passed.


----

# Silver SQL Write Cell
Target:
silver.sensor_observation_features

Purpose:
Write the final Silver dataframe into Postgres as feature JSON plus quality, metadata, preserving dataset/run lineage.



In [176]:

WRITE_TO_POSTGRES = True

if WRITE_TO_POSTGRES:

    silver_eda_sql_summary_dataframe = log_silver_eda_sql(
        engine=engine,
        capstone_schema=CAPSTONE_SCHEMA,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        notebook_globals=globals(),
        dataset_name=globals().get("DATASET_NAME", DATASET_ID),
    )

    display(silver_eda_sql_summary_dataframe)

else:
    print("Postgres write skipped.")

Using dataframe: dataframe -> 220,320 rows x 73 columns
Upserted pipeline run metadata: run__001__silver_eda
Logged DQ event: silver.silver_eda_notebook | silver_eda_profile | pass


,layer_name,table_name,check_name,check_status,row_count,created_at_utc
0,silver,silver_eda_notebook,silver_eda_profile,pass,220320,2026-06-14 22:04:40.972433+00:00
1,silver,silver_eda_notebook,silver_eda_profile,pass,220320,2026-06-14 21:21:01.629726+00:00


## Summary

This notebook preserves the current analytical behavior while documenting the role of the Silver step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

Silver 02a uses this output to build the profiled subsets and clean-normal construction inputs.
